# Unified Nearable rPPG Pipeline

Upload one video. The notebook classifies the head-motion type (`stable`, `left_right`, or `zigzag`) and then runs the matching final rPPG pipeline.


In [ ]:
# =========================
# Unified Nearable rPPG Pipeline
# Combines:
# 1) Head motion classification
# 2) Fixed-video rPPG pipeline
# 3) Left-right rPPG pipeline
# 4) ZigZag rPPG pipeline
# =========================

# -*- coding: utf-8 -*-
"""Head Tracking.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1BDseC0mU-XSyCYk8vO1UoHkQLkl6vEv2
"""

# =========================
# Single Video Head Motion Classifier for Google Colab
#
# Classes:
# - stable
# - left_right
# - zigzag
#
# Upload one video.
# The code predicts the head motion type.
#
# Updated version:
# - Trims first/last seconds for classification
# - Handles detector drift as stable
# - Removes tracking spikes/outliers
# - Detects strong left-right even if x_direction_changes is zero
# - Detects zigzag/periodic mixed motion when vertical oscillation is repeated
# - Updated PERIODIC_ZIGZAG_MIN_Y from 0.080 to 0.060
# =========================

import cv2
import numpy as np
import matplotlib.pyplot as plt

from scipy.signal import savgol_filter
from google.colab import files
from google.colab.patches import cv2_imshow



# =========================
# Settings
# =========================
SHOW_PREVIEW_FRAMES = True
DETECT_EVERY_N_FRAMES = 10
MIN_FACE_SIZE = (90, 90)

# Ignore beginning and ending parts for classification.
# This helps remove detector initialization drift and final tracking jumps.
CLASSIFICATION_TRIM_SECONDS = 4.0

# Stable thresholds
STABLE_X_THRESHOLD = 0.110
STABLE_Y_THRESHOLD = 0.110
STABLE_TOTAL_THRESHOLD = 0.140

# Drift-stable rule
# If there is no real repeated back-and-forth motion,
# classify as stable even if there is moderate tracking drift.
DRIFT_STABLE_TOTAL_THRESHOLD = 0.220
DRIFT_STABLE_MAX_X_CHANGES = 1
DRIFT_STABLE_MAX_Y_CHANGES = 2

# Left-right thresholds
LEFT_RIGHT_MIN_X = 0.120
LEFT_RIGHT_MAX_Y_TO_X_RATIO = 0.60
LEFT_RIGHT_MIN_X_CHANGES = 1

# Strong left-right rule
# This catches smooth left-right cases where x_direction_changes becomes zero.
STRONG_LEFT_RIGHT_MIN_X = 0.300
STRONG_LEFT_RIGHT_MAX_Y_TO_X_RATIO = 0.250

# Zigzag thresholds
ZIGZAG_MIN_X = 0.090
ZIGZAG_MIN_Y = 0.090
ZIGZAG_MIN_Y_TO_X_RATIO = 0.55
ZIGZAG_MIN_TOTAL_DIRECTION_CHANGES = 3

# Periodic/zigzag mixed motion thresholds
# This catches cases where x is dominant but y is oscillating repeatedly.
PERIODIC_ZIGZAG_MIN_X = 0.250
PERIODIC_ZIGZAG_MIN_Y = 0.060
PERIODIC_ZIGZAG_MIN_Y_CHANGES = 5
PERIODIC_ZIGZAG_MIN_TOTAL_CHANGES = 7

# Spike removal setting
SPIKE_Z_THRESHOLD = 3.5


# =========================
# Helper functions
# =========================
def largest_face(faces):
    if len(faces) == 0:
        return None

    faces = sorted(faces, key=lambda f: f[2] * f[3], reverse=True)
    return faces[0]


def keep_box_inside_frame(box, frame_shape):
    x, y, w, h = box
    height, width = frame_shape[:2]

    x = max(0, int(x))
    y = max(0, int(y))
    w = max(1, min(int(w), width - x))
    h = max(1, min(int(h), height - y))

    return x, y, w, h


def expand_box(box, frame_shape, scale=1.15):
    x, y, w, h = box

    cx = x + w / 2.0
    cy = y + h / 2.0

    new_w = w * scale
    new_h = h * scale

    new_x = cx - new_w / 2.0
    new_y = cy - new_h / 2.0

    return keep_box_inside_frame((new_x, new_y, new_w, new_h), frame_shape)


def box_center(box):
    x, y, w, h = box
    return np.array([x + w / 2.0, y + h / 2.0], dtype=float)


def smooth_signal(signal):
    signal = np.asarray(signal, dtype=float)

    if len(signal) < 9:
        return signal

    window = min(21, len(signal))

    if window % 2 == 0:
        window -= 1

    if window < 7:
        return signal

    try:
        return savgol_filter(signal, window_length=window, polyorder=2)
    except Exception:
        return signal


def remove_spikes(signal, z_threshold=3.5):
    signal = np.asarray(signal, dtype=float)

    if len(signal) < 9:
        return signal

    cleaned = signal.copy()

    median_value = np.median(cleaned)
    mad = np.median(np.abs(cleaned - median_value)) + 1e-8

    robust_z = 0.6745 * (cleaned - median_value) / mad
    spike_mask = np.abs(robust_z) > z_threshold

    if np.sum(spike_mask) == 0:
        return cleaned

    valid_indices = np.where(~spike_mask)[0]
    spike_indices = np.where(spike_mask)[0]

    if len(valid_indices) < 5:
        return cleaned

    cleaned[spike_indices] = np.interp(
        spike_indices,
        valid_indices,
        cleaned[valid_indices]
    )

    return cleaned


def robust_range(signal, low=5, high=95):
    signal = np.asarray(signal, dtype=float)

    if len(signal) == 0:
        return 0.0

    return float(np.percentile(signal, high) - np.percentile(signal, low))


def count_direction_changes(signal, min_delta=0.004):
    signal = np.asarray(signal, dtype=float)

    if len(signal) < 5:
        return 0

    diff = np.diff(signal)

    # Remove tiny fluctuations
    diff[np.abs(diff) < min_delta] = 0.0

    signs = np.sign(diff)
    signs = signs[signs != 0]

    if len(signs) < 2:
        return 0

    return int(np.sum(signs[1:] != signs[:-1]))


def get_tracking_points(gray, face_box):
    x, y, w, h = expand_box(face_box, gray.shape, scale=1.10)

    mask = np.zeros_like(gray)
    mask[y:y+h, x:x+w] = 255

    points = cv2.goodFeaturesToTrack(
        gray,
        maxCorners=100,
        qualityLevel=0.01,
        minDistance=7,
        blockSize=7,
        mask=mask
    )

    return points


# =========================
# Extract head motion
# =========================
def extract_head_motion(video_path):
    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        raise RuntimeError("Video could not be opened.")

    fps = cap.get(cv2.CAP_PROP_FPS)

    if fps is None or fps <= 0:
        fps = 30.0

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duration_sec = total_frames / fps if fps > 0 else 0

    face_cascade = cv2.CascadeClassifier(
        cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
    )

    centers = []
    face_sizes = []
    frame_indices = []
    preview_frames = []

    prev_gray = None
    prev_points = None

    last_face = None
    last_center = None

    frame_index = 0

    while True:
        ret, frame = cap.read()

        if not ret:
            break

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        need_detection = (
            last_face is None or
            prev_points is None or
            len(prev_points) < 10 or
            frame_index % DETECT_EVERY_N_FRAMES == 0
        )

        if need_detection:
            faces = face_cascade.detectMultiScale(
                gray,
                scaleFactor=1.1,
                minNeighbors=4,
                minSize=MIN_FACE_SIZE
            )

            detected_face = largest_face(faces)

            if detected_face is not None:
                last_face = detected_face
                last_center = box_center(last_face)
                prev_points = get_tracking_points(gray, last_face)

        current_center = None

        if prev_gray is not None and prev_points is not None and len(prev_points) >= 5:
            next_points, status, error = cv2.calcOpticalFlowPyrLK(
                prev_gray,
                gray,
                prev_points,
                None,
                winSize=(21, 21),
                maxLevel=3,
                criteria=(
                    cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT,
                    30,
                    0.01
                )
            )

            if next_points is not None and status is not None:
                status = status.reshape(-1)

                good_new = next_points[status == 1]
                good_old = prev_points[status == 1]

                if len(good_new) >= 5:
                    median_shift = np.median(
                        good_new.reshape(-1, 2) - good_old.reshape(-1, 2),
                        axis=0
                    )

                    if last_center is not None:
                        current_center = last_center + median_shift
                        last_center = current_center

                    prev_points = good_new.reshape(-1, 1, 2)

        if current_center is None and last_face is not None:
            current_center = box_center(last_face)
            last_center = current_center

        if current_center is not None and last_face is not None:
            x, y, w, h = last_face
            face_size = max(float(w), float(h))

            centers.append(current_center)
            face_sizes.append(face_size)
            frame_indices.append(frame_index)

            if SHOW_PREVIEW_FRAMES and len(preview_frames) < 5:
                preview = frame.copy()

                x, y, w, h = keep_box_inside_frame(last_face, frame.shape)
                cx = int(current_center[0])
                cy = int(current_center[1])

                cv2.rectangle(
                    preview,
                    (x, y),
                    (x + w, y + h),
                    (0, 255, 0),
                    2
                )

                cv2.circle(
                    preview,
                    (cx, cy),
                    5,
                    (0, 0, 255),
                    -1
                )

                cv2.putText(
                    preview,
                    "Tracked head center",
                    (x, max(0, y - 8)),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6,
                    (0, 255, 0),
                    2
                )

                preview_frames.append(preview)

        prev_gray = gray.copy()
        frame_index += 1

    cap.release()

    centers = np.asarray(centers, dtype=float)
    face_sizes = np.asarray(face_sizes, dtype=float)
    frame_indices = np.asarray(frame_indices, dtype=int)

    if len(centers) < 30:
        raise RuntimeError(
            "Not enough valid face tracking samples. "
            "Use a clearer video with better lighting and visible face."
        )

    median_face_size = float(np.median(face_sizes))

    if median_face_size <= 1:
        median_face_size = 1.0

    times = frame_indices / fps

    x_raw = centers[:, 0]
    y_raw = centers[:, 1]

    x_smooth = smooth_signal(x_raw)
    y_smooth = smooth_signal(y_raw)

    x_norm = (x_smooth - np.median(x_smooth)) / median_face_size
    y_norm = (y_smooth - np.median(y_smooth)) / median_face_size

    data = {
        "fps": fps,
        "total_frames": total_frames,
        "duration_sec": duration_sec,
        "valid_samples": len(centers),
        "valid_ratio": len(centers) / max(1, frame_index),
        "median_face_size": median_face_size,
        "times": times,
        "x_norm": x_norm,
        "y_norm": y_norm,
        "preview_frames": preview_frames
    }

    return data


# =========================
# Feature extraction
# =========================
def compute_features(data):
    x_all = np.asarray(data["x_norm"], dtype=float)
    y_all = np.asarray(data["y_norm"], dtype=float)
    times_all = np.asarray(data["times"], dtype=float)

    # Trim first and last seconds to reduce detector initialization/end drift
    if len(times_all) > 0:
        relative_time = times_all - times_all[0]
        duration = relative_time[-1]

        keep_mask = (
            (relative_time >= CLASSIFICATION_TRIM_SECONDS) &
            (relative_time <= duration - CLASSIFICATION_TRIM_SECONDS)
        )

        if np.sum(keep_mask) >= 30:
            x = x_all[keep_mask]
            y = y_all[keep_mask]
            used_trim = True
        else:
            x = x_all
            y = y_all
            used_trim = False
    else:
        x = x_all
        y = y_all
        used_trim = False

    # Remove tracking spikes/outliers before feature calculation
    x = remove_spikes(x, z_threshold=SPIKE_Z_THRESHOLD)
    y = remove_spikes(y, z_threshold=SPIKE_Z_THRESHOLD)

    # Smooth again after spike interpolation
    x = smooth_signal(x)
    y = smooth_signal(y)

    x_amp = robust_range(x)
    y_amp = robust_range(y)
    total_amp = float(np.sqrt(x_amp ** 2 + y_amp ** 2))

    y_to_x_ratio = float(y_amp / (x_amp + 1e-8))
    x_to_y_ratio = float(x_amp / (y_amp + 1e-8))

    x_changes = count_direction_changes(x)
    y_changes = count_direction_changes(y)

    path_length = float(
        np.sum(
            np.sqrt(
                np.diff(x) ** 2 +
                np.diff(y) ** 2
            )
        )
    )

    features = {
        "x_amp": x_amp,
        "y_amp": y_amp,
        "total_amp": total_amp,
        "y_to_x_ratio": y_to_x_ratio,
        "x_to_y_ratio": x_to_y_ratio,
        "x_direction_changes": x_changes,
        "y_direction_changes": y_changes,
        "total_direction_changes": x_changes + y_changes,
        "path_length": path_length,
        "used_trim": used_trim
    }

    return features


# =========================
# Classification
# =========================
def classify_motion(features):
    x_amp = features["x_amp"]
    y_amp = features["y_amp"]
    total_amp = features["total_amp"]
    y_to_x_ratio = features["y_to_x_ratio"]

    x_changes = features["x_direction_changes"]
    y_changes = features["y_direction_changes"]
    total_changes = features["total_direction_changes"]

    # Case 1: strong stable
    # Small motion in both x and y.
    if (
        x_amp < STABLE_X_THRESHOLD and
        y_amp < STABLE_Y_THRESHOLD and
        total_amp < STABLE_TOTAL_THRESHOLD
    ):
        predicted_class = "stable"
        confidence = 0.90

    # Case 2: zigzag / periodic mixed motion
    # If vertical direction changes are high, it is not pure left-right.
    # This catches periodic/zigzag videos where x is dominant but y oscillates repeatedly.
    elif (
        x_amp >= PERIODIC_ZIGZAG_MIN_X and
        y_amp >= PERIODIC_ZIGZAG_MIN_Y and
        y_changes >= PERIODIC_ZIGZAG_MIN_Y_CHANGES and
        total_changes >= PERIODIC_ZIGZAG_MIN_TOTAL_CHANGES
    ):
        predicted_class = "zigzag"

        vertical_activity = min(y_changes / 10.0, 1.0)
        y_strength = min(y_amp / 0.15, 1.0)

        confidence = (
            0.70
            + 0.15 * vertical_activity
            + 0.10 * y_strength
        )

        confidence = min(confidence, 0.95)

    # Case 3: strong left-right without relying on direction changes.
    # This catches smooth left-right cases where x_direction_changes becomes zero.
    elif (
        x_amp >= STRONG_LEFT_RIGHT_MIN_X and
        y_to_x_ratio <= STRONG_LEFT_RIGHT_MAX_Y_TO_X_RATIO
    ):
        predicted_class = "left_right"

        dominance = x_amp / (y_amp + 1e-8)
        motion_strength = min(x_amp / 0.50, 1.0)

        confidence = (
            0.70
            + 0.15 * min(dominance / 5.0, 1.0)
            + 0.10 * motion_strength
        )

        confidence = min(confidence, 0.95)

    # Case 4: stable with drift
    # This handles videos where the detector slowly drifts or jumps,
    # but there is no real repeated head motion.
    elif (
        total_amp < DRIFT_STABLE_TOTAL_THRESHOLD and
        x_changes <= DRIFT_STABLE_MAX_X_CHANGES and
        y_changes <= DRIFT_STABLE_MAX_Y_CHANGES
    ):
        predicted_class = "stable"
        confidence = 0.75

    # Case 5: normal left-right
    # Real left-right usually has dominant x motion and at least one direction change.
    elif (
        x_amp >= LEFT_RIGHT_MIN_X and
        y_to_x_ratio <= LEFT_RIGHT_MAX_Y_TO_X_RATIO and
        x_changes >= LEFT_RIGHT_MIN_X_CHANGES
    ):
        predicted_class = "left_right"

        dominance = x_amp / (y_amp + 1e-8)
        confidence = 0.65 + 0.25 * min(dominance / 3.0, 1.0)
        confidence = min(confidence, 0.95)

    # Case 6: general zigzag
    # Zigzag should have both x and y motion plus repeated direction changes.
    elif (
        x_amp >= ZIGZAG_MIN_X and
        y_amp >= ZIGZAG_MIN_Y and
        y_to_x_ratio >= ZIGZAG_MIN_Y_TO_X_RATIO and
        total_changes >= ZIGZAG_MIN_TOTAL_DIRECTION_CHANGES
    ):
        predicted_class = "zigzag"

        balance = 1.0 - min(
            abs(x_amp - y_amp) / (max(x_amp, y_amp) + 1e-8),
            1.0
        )

        change_score = min(total_changes / 8.0, 1.0)
        confidence = 0.60 + 0.20 * balance + 0.15 * change_score
        confidence = min(confidence, 0.95)

    # Fallback cases
    else:
        if total_amp < 0.170 and total_changes <= 2:
            predicted_class = "stable"
            confidence = 0.65

        # If horizontal motion is very small, vertical spikes alone should not make it zigzag.
        elif x_amp < 0.050 and y_amp < 0.220:
            predicted_class = "stable"
            confidence = 0.65

        # Periodic/zigzag fallback:
        # Repeated vertical oscillation should not be classified as pure left-right.
        elif (
            y_amp >= PERIODIC_ZIGZAG_MIN_Y and
            y_changes >= PERIODIC_ZIGZAG_MIN_Y_CHANGES and
            total_changes >= PERIODIC_ZIGZAG_MIN_TOTAL_CHANGES
        ):
            predicted_class = "zigzag"
            confidence = 0.70

        # Additional fallback for horizontal-dominant motion
        elif x_amp >= 0.250 and y_to_x_ratio <= 0.300:
            predicted_class = "left_right"
            confidence = 0.70

        elif y_to_x_ratio < 0.70 and x_changes >= 1:
            predicted_class = "left_right"
            confidence = 0.65

        elif total_changes >= 3:
            predicted_class = "zigzag"
            confidence = 0.65

        else:
            predicted_class = "stable"
            confidence = 0.60

    return predicted_class, confidence




# =========================
# Embedded final rPPG pipelines
# =========================
FIXED_PIPELINE_CODE = '# =========================\n# Final Smart Multi-ROI + POS rPPG for Google Colab\n# Offline video version\n# No real HR required\n#\n# Features:\n# - Multi-ROI RGB extraction\n# - POS rPPG\n# - Zero-padded FFT\n# - Parabolic peak interpolation\n# - Window stability\n# - Green/POS consensus\n# - Peak agreement correction\n# - Safer adaptive high-HR rescue rule\n# =========================\n\nimport cv2\nimport numpy as np\nimport matplotlib.pyplot as plt\nfrom scipy.signal import butter, filtfilt, find_peaks, detrend\nfrom google.colab.patches import cv2_imshow\nfrom google.colab import files\n\n\n# Video path is provided by the master classifier.\nprint("Using video:", video_path)\n\n\n# =========================\n# Global settings\n# =========================\nMIN_BPM = 60\nMAX_BPM = 90\n\nTRIM_START_SECONDS = 3.0\n\nPOS_WINDOW_SECONDS = 1.6\n\nWINDOW_SECONDS = 12.0\nWINDOW_STEP_SECONDS = 2.0\n\nFFT_TOP_N_PEAKS = 8\nFFT_ZERO_PADDING_FACTOR = 8\n\n\n# =========================\n# Basic signal functions\n# =========================\ndef normalize_signal(signal):\n    signal = np.asarray(signal, dtype=float)\n    mean = np.mean(signal)\n    std = np.std(signal)\n\n    if std < 1e-8:\n        return signal - mean\n\n    return (signal - mean) / std\n\n\ndef bandpass_filter(signal, fs, min_bpm=60, max_bpm=90, order=4):\n    signal = np.asarray(signal, dtype=float)\n\n    low_hz = min_bpm / 60.0\n    high_hz = max_bpm / 60.0\n\n    try:\n        b, a = butter(order, [low_hz, high_hz], btype="bandpass", fs=fs)\n\n        padlen = 3 * max(len(a), len(b))\n        if len(signal) <= padlen:\n            return signal\n\n        return filtfilt(b, a, signal)\n\n    except Exception as e:\n        print("[Filter warning]", e)\n        return signal\n\n\ndef trim_signal_start(signal, fs, trim_start_seconds=3.0):\n    signal = np.asarray(signal, dtype=float)\n    trim_samples = int(trim_start_seconds * fs)\n\n    if len(signal) > trim_samples + int(fs * 6):\n        return signal[trim_samples:]\n\n    return signal\n\n\ndef next_power_of_two(n):\n    return int(2 ** np.ceil(np.log2(max(1, n))))\n\n\ndef parabolic_interpolation(y, index):\n    if index <= 0 or index >= len(y) - 1:\n        return 0.0\n\n    y0 = y[index - 1]\n    y1 = y[index]\n    y2 = y[index + 1]\n\n    denominator = y0 - 2.0 * y1 + y2\n\n    if abs(denominator) < 1e-12:\n        return 0.0\n\n    delta = 0.5 * (y0 - y2) / denominator\n    delta = float(np.clip(delta, -0.5, 0.5))\n\n    return delta\n\n\ndef compute_fft_spectrum(\n    signal,\n    fs,\n    min_bpm=60,\n    max_bpm=90,\n    trim_start_seconds=3.0,\n    zero_padding_factor=8\n):\n    signal = np.asarray(signal, dtype=float)\n    signal = trim_signal_start(signal, fs, trim_start_seconds)\n\n    if len(signal) < int(fs * 6):\n        return None, None\n\n    signal = signal - np.mean(signal)\n    signal = signal * np.hanning(len(signal))\n\n    n_fft = next_power_of_two(len(signal)) * zero_padding_factor\n\n    freqs = np.fft.rfftfreq(n_fft, d=1.0 / fs)\n    spectrum = np.abs(np.fft.rfft(signal, n=n_fft))\n\n    bpm_axis = freqs * 60.0\n    mask = (bpm_axis >= min_bpm) & (bpm_axis <= max_bpm)\n\n    if not np.any(mask):\n        return None, None\n\n    return bpm_axis[mask], spectrum[mask]\n\n\ndef refined_peak_bpm(bpm_axis, spectrum, peak_index):\n    if bpm_axis is None or spectrum is None:\n        return None\n\n    if len(bpm_axis) < 2:\n        return float(bpm_axis[peak_index])\n\n    bpm_step = bpm_axis[1] - bpm_axis[0]\n\n    safe_spectrum = np.maximum(spectrum, 1e-12)\n    log_spectrum = np.log(safe_spectrum)\n\n    delta = parabolic_interpolation(log_spectrum, peak_index)\n    refined_bpm = bpm_axis[peak_index] + delta * bpm_step\n\n    return float(refined_bpm)\n\n\ndef get_fft_candidates(\n    signal,\n    fs,\n    min_bpm=60,\n    max_bpm=90,\n    trim_start_seconds=3.0,\n    top_n_peaks=8\n):\n    bpm_axis, spectrum = compute_fft_spectrum(\n        signal,\n        fs,\n        min_bpm=min_bpm,\n        max_bpm=max_bpm,\n        trim_start_seconds=trim_start_seconds,\n        zero_padding_factor=FFT_ZERO_PADDING_FACTOR\n    )\n\n    if bpm_axis is None or spectrum is None or len(spectrum) == 0:\n        return []\n\n    max_mag = np.max(spectrum)\n\n    if max_mag < 1e-8:\n        return []\n\n    peak_indices, _ = find_peaks(\n        spectrum,\n        distance=2,\n        prominence=0.03 * max_mag\n    )\n\n    if len(peak_indices) == 0:\n        peak_indices = np.array([int(np.argmax(spectrum))])\n\n    peak_mag = spectrum[peak_indices]\n\n    order = np.argsort(peak_mag)[::-1]\n    order = order[:top_n_peaks]\n\n    candidates = []\n\n    for idx in order:\n        peak_index = int(peak_indices[idx])\n        bpm = refined_peak_bpm(bpm_axis, spectrum, peak_index)\n\n        candidates.append({\n            "bpm": float(bpm),\n            "magnitude": float(spectrum[peak_index]),\n            "norm_magnitude": float(spectrum[peak_index] / max_mag)\n        })\n\n    return candidates\n\n\ndef estimate_windowed_fft(\n    signal,\n    fs,\n    min_bpm=60,\n    max_bpm=90,\n    trim_start_seconds=3.0,\n    window_seconds=12.0,\n    step_seconds=2.0\n):\n    signal = np.asarray(signal, dtype=float)\n    signal = trim_signal_start(signal, fs, trim_start_seconds)\n\n    window_len = int(window_seconds * fs)\n    step_len = int(step_seconds * fs)\n\n    if len(signal) < window_len:\n        return []\n\n    estimates = []\n\n    for start in range(0, len(signal) - window_len + 1, step_len):\n        end = start + window_len\n        segment = signal[start:end]\n\n        bpm_axis, spectrum = compute_fft_spectrum(\n            segment,\n            fs,\n            min_bpm=min_bpm,\n            max_bpm=max_bpm,\n            trim_start_seconds=0.0,\n            zero_padding_factor=FFT_ZERO_PADDING_FACTOR\n        )\n\n        if bpm_axis is None or spectrum is None or len(spectrum) == 0:\n            continue\n\n        max_mag = np.max(spectrum)\n        mean_mag = np.mean(spectrum) + 1e-8\n\n        if max_mag < 1e-8:\n            continue\n\n        best_idx = int(np.argmax(spectrum))\n        best_bpm = refined_peak_bpm(bpm_axis, spectrum, best_idx)\n        quality = float(max_mag / mean_mag)\n\n        estimates.append({\n            "bpm": float(best_bpm),\n            "quality": quality,\n            "start_sec": start / fs,\n            "end_sec": end / fs\n        })\n\n    return estimates\n\n\ndef weighted_mean(values, weights):\n    values = np.asarray(values, dtype=float)\n    weights = np.asarray(weights, dtype=float)\n\n    if len(values) == 0:\n        return None\n\n    if np.sum(weights) < 1e-8:\n        return float(np.mean(values))\n\n    return float(np.sum(values * weights) / np.sum(weights))\n\n\ndef robust_window_hr(window_estimates):\n    if len(window_estimates) == 0:\n        return None, None, None\n\n    bpms = np.asarray([item["bpm"] for item in window_estimates], dtype=float)\n    qualities = np.asarray([item["quality"] for item in window_estimates], dtype=float)\n\n    median_hr = float(np.median(bpms))\n    mad = float(np.median(np.abs(bpms - median_hr)))\n\n    if len(bpms) >= 3:\n        keep_mask = np.abs(bpms - median_hr) <= max(4.0, 1.5 * mad)\n        kept_bpms = bpms[keep_mask]\n        kept_qualities = qualities[keep_mask]\n    else:\n        kept_bpms = bpms\n        kept_qualities = qualities\n\n    if len(kept_bpms) == 0:\n        kept_bpms = bpms\n        kept_qualities = qualities\n\n    stable_hr = weighted_mean(kept_bpms, kept_qualities)\n\n    return stable_hr, median_hr, mad\n\n\ndef summarize_channel_fft(\n    signal,\n    fs,\n    min_bpm=60,\n    max_bpm=90,\n    trim_start_seconds=3.0,\n    top_n_peaks=8,\n    window_seconds=12.0,\n    step_seconds=2.0\n):\n    candidates = get_fft_candidates(\n        signal,\n        fs,\n        min_bpm=min_bpm,\n        max_bpm=max_bpm,\n        trim_start_seconds=trim_start_seconds,\n        top_n_peaks=top_n_peaks\n    )\n\n    window_estimates = estimate_windowed_fft(\n        signal,\n        fs,\n        min_bpm=min_bpm,\n        max_bpm=max_bpm,\n        trim_start_seconds=trim_start_seconds,\n        window_seconds=window_seconds,\n        step_seconds=step_seconds\n    )\n\n    full_best = None\n    if len(candidates) > 0:\n        full_best = candidates[0]["bpm"]\n\n    window_hr, window_median, window_mad = robust_window_hr(window_estimates)\n\n    if window_hr is None:\n        window_hr = full_best\n\n    window_qualities = [item["quality"] for item in window_estimates]\n\n    if window_mad is None:\n        stability_score = 0.0\n    else:\n        stability_score = 1.0 / (1.0 + window_mad)\n\n    if len(window_qualities) > 0:\n        quality_score = float(np.median(window_qualities))\n    else:\n        quality_score = 0.0\n\n    if len(candidates) >= 2:\n        peak_ratio = candidates[0]["magnitude"] / (candidates[1]["magnitude"] + 1e-8)\n    else:\n        peak_ratio = 1.0\n\n    reliability = float(\n        0.45 * stability_score +\n        0.35 * min(quality_score / 5.0, 1.0) +\n        0.20 * min(peak_ratio / 2.0, 1.0)\n    )\n\n    return {\n        "candidates": candidates,\n        "window_estimates": window_estimates,\n        "full_best": full_best,\n        "window_hr": window_hr,\n        "window_median": window_median,\n        "window_mad": window_mad,\n        "quality_score": quality_score,\n        "peak_ratio": peak_ratio,\n        "reliability": reliability\n    }\n\n\ndef build_consensus_hr(green_info, pos_info):\n    green_window = green_info["window_hr"]\n    pos_window = pos_info["window_hr"]\n    green_full = green_info["full_best"]\n    pos_full = pos_info["full_best"]\n\n    values = []\n    weights = []\n\n    if green_window is not None:\n        values.append(green_window)\n        weights.append(1.0 + green_info["reliability"])\n\n    if pos_window is not None:\n        values.append(pos_window)\n        weights.append(1.0 + pos_info["reliability"])\n\n    if green_full is not None:\n        values.append(green_full)\n        weights.append(0.45 + 0.5 * green_info["reliability"])\n\n    if pos_full is not None:\n        values.append(pos_full)\n        weights.append(0.45 + 0.5 * pos_info["reliability"])\n\n    if len(values) == 0:\n        return None\n\n    if green_window is not None and pos_window is not None:\n        diff = abs(green_window - pos_window)\n\n        if diff <= 4.0:\n            return weighted_mean(\n                [green_window, pos_window],\n                [1.0 + green_info["reliability"], 1.0 + pos_info["reliability"]]\n            )\n\n        if green_info["reliability"] > pos_info["reliability"] + 0.15:\n            return float(green_window)\n\n        if pos_info["reliability"] > green_info["reliability"] + 0.15:\n            return float(pos_window)\n\n    return weighted_mean(values, weights)\n\n\ndef choose_candidate_near_consensus(channel_info, consensus_hr):\n    if consensus_hr is None:\n        if channel_info["window_hr"] is not None:\n            return channel_info["window_hr"]\n        return channel_info["full_best"]\n\n    candidate_bpms = []\n    candidate_weights = []\n\n    for candidate in channel_info["candidates"]:\n        candidate_bpms.append(candidate["bpm"])\n        candidate_weights.append(candidate["norm_magnitude"])\n\n    if channel_info["window_hr"] is not None:\n        candidate_bpms.append(channel_info["window_hr"])\n        candidate_weights.append(0.95)\n\n    if channel_info["full_best"] is not None:\n        candidate_bpms.append(channel_info["full_best"])\n        candidate_weights.append(0.75)\n\n    if len(candidate_bpms) == 0:\n        return consensus_hr\n\n    candidate_bpms = np.asarray(candidate_bpms, dtype=float)\n    candidate_weights = np.asarray(candidate_weights, dtype=float)\n\n    distances = np.abs(candidate_bpms - consensus_hr)\n    scores = distances - 2.0 * candidate_weights\n\n    best_idx = int(np.argmin(scores))\n    selected = float(candidate_bpms[best_idx])\n\n    if abs(selected - consensus_hr) > 6.0:\n        selected = float(consensus_hr)\n\n    return selected\n\n\ndef estimate_smart_fft_pair(\n    green_signal,\n    pos_signal,\n    fs,\n    min_bpm=60,\n    max_bpm=90,\n    trim_start_seconds=3.0\n):\n    green_info = summarize_channel_fft(\n        green_signal,\n        fs,\n        min_bpm=min_bpm,\n        max_bpm=max_bpm,\n        trim_start_seconds=trim_start_seconds,\n        top_n_peaks=FFT_TOP_N_PEAKS,\n        window_seconds=WINDOW_SECONDS,\n        step_seconds=WINDOW_STEP_SECONDS\n    )\n\n    pos_info = summarize_channel_fft(\n        pos_signal,\n        fs,\n        min_bpm=min_bpm,\n        max_bpm=max_bpm,\n        trim_start_seconds=trim_start_seconds,\n        top_n_peaks=FFT_TOP_N_PEAKS,\n        window_seconds=WINDOW_SECONDS,\n        step_seconds=WINDOW_STEP_SECONDS\n    )\n\n    consensus_hr = build_consensus_hr(green_info, pos_info)\n\n    green_hr = choose_candidate_near_consensus(green_info, consensus_hr)\n    pos_hr = choose_candidate_near_consensus(pos_info, consensus_hr)\n\n    return green_hr, pos_hr, consensus_hr, green_info, pos_info\n\n\ndef estimate_hr_peaks(signal, fs, min_bpm=60, max_bpm=90):\n    signal = np.asarray(signal, dtype=float)\n\n    if len(signal) < int(fs * 6):\n        return None\n\n    signal = signal - np.mean(signal)\n\n    min_distance = max(1, int(fs * 60.0 / max_bpm))\n    prominence = max(0.35 * np.std(signal), 0.15)\n\n    peaks, _ = find_peaks(\n        signal,\n        distance=min_distance,\n        prominence=prominence\n    )\n\n    if len(peaks) < 2:\n        return None\n\n    intervals = np.diff(peaks) / fs\n    bpm_values = 60.0 / intervals\n\n    bpm_values = bpm_values[\n        (bpm_values >= min_bpm) & (bpm_values <= max_bpm)\n    ]\n\n    if len(bpm_values) == 0:\n        return None\n\n    return float(np.median(bpm_values))\n\n\ndef adaptive_final_correction(\n    green_info,\n    pos_info,\n    hr_green_fft,\n    hr_pos_fft,\n    hr_consensus,\n    hr_green_peak,\n    hr_pos_peak\n):\n    def get_strong_high_candidate(channel_info, low_bpm=78.0, high_bpm=86.5, min_norm=0.65):\n        valid_candidates = []\n\n        for candidate in channel_info["candidates"]:\n            bpm = candidate["bpm"]\n            norm = candidate["norm_magnitude"]\n\n            if low_bpm <= bpm <= high_bpm and norm >= min_norm:\n                valid_candidates.append(candidate)\n\n        if len(valid_candidates) == 0:\n            return None\n\n        valid_candidates = sorted(\n            valid_candidates,\n            key=lambda c: c["norm_magnitude"],\n            reverse=True\n        )\n\n        return float(valid_candidates[0]["bpm"])\n\n    def is_full_best_high(channel_info, low_bpm=78.0, high_bpm=86.5):\n        full_best = channel_info["full_best"]\n\n        if full_best is None:\n            return False\n\n        return low_bpm <= full_best <= high_bpm\n\n    peak_guidance_hr = None\n\n    green_high_candidate = get_strong_high_candidate(green_info)\n    pos_high_candidate = get_strong_high_candidate(pos_info)\n\n    green_full_is_high = is_full_best_high(green_info)\n    pos_full_is_high = is_full_best_high(pos_info)\n\n    if green_high_candidate is not None and pos_high_candidate is not None:\n        high_candidate_diff = abs(green_high_candidate - pos_high_candidate)\n        high_candidate_consensus = float(np.median([green_high_candidate, pos_high_candidate]))\n\n        current_values = [\n            value for value in [hr_green_fft, hr_pos_fft, hr_consensus]\n            if value is not None\n        ]\n\n        if len(current_values) > 0:\n            current_consensus = float(np.median(current_values))\n        else:\n            current_consensus = None\n\n        high_full_support = green_full_is_high or pos_full_is_high\n\n        if (\n            high_candidate_diff <= 4.0 and\n            current_consensus is not None and\n            high_candidate_consensus - current_consensus >= 5.0 and\n            high_full_support\n        ):\n            hr_green_fft = float(green_high_candidate)\n            hr_pos_fft = float(pos_high_candidate)\n            hr_consensus = float(np.median([hr_green_fft, hr_pos_fft]))\n            peak_guidance_hr = None\n\n            return hr_green_fft, hr_pos_fft, hr_consensus, peak_guidance_hr\n\n    if hr_green_peak is not None and hr_pos_peak is not None:\n        peak_diff = abs(hr_green_peak - hr_pos_peak)\n\n        if peak_diff <= 3.0:\n            peak_guidance_hr = float(np.median([hr_green_peak, hr_pos_peak]))\n\n            if hr_green_fft is not None and abs(hr_green_fft - peak_guidance_hr) > 2.0:\n                hr_green_fft = peak_guidance_hr\n\n            if hr_pos_fft is not None and abs(hr_pos_fft - peak_guidance_hr) > 2.0:\n                hr_pos_fft = peak_guidance_hr\n\n            if hr_green_fft is not None and hr_pos_fft is not None:\n                hr_consensus = float(np.median([hr_green_fft, hr_pos_fft]))\n\n        else:\n            peak_guidance_hr = float(np.median([hr_green_peak, hr_pos_peak]))\n\n    elif hr_pos_peak is not None:\n        peak_guidance_hr = float(hr_pos_peak)\n\n    elif hr_green_peak is not None:\n        peak_guidance_hr = float(hr_green_peak)\n\n    return hr_green_fft, hr_pos_fft, hr_consensus, peak_guidance_hr\n\n\n# =========================\n# ROI functions\n# =========================\ndef keep_roi_inside_frame(roi, frame_shape):\n    x, y, w, h = roi\n    height, width = frame_shape[:2]\n\n    x = max(0, x)\n    y = max(0, y)\n    w = max(1, min(w, width - x))\n    h = max(1, min(h, height - y))\n\n    return x, y, w, h\n\n\ndef get_multi_rois(face, frame_shape):\n    x, y, w, h = face\n\n    rois = {}\n\n    rois["forehead"] = (\n        x + int(0.30 * w),\n        y + int(0.08 * h),\n        int(0.40 * w),\n        int(0.18 * h)\n    )\n\n    rois["left_cheek"] = (\n        x + int(0.18 * w),\n        y + int(0.48 * h),\n        int(0.22 * w),\n        int(0.20 * h)\n    )\n\n    rois["right_cheek"] = (\n        x + int(0.60 * w),\n        y + int(0.48 * h),\n        int(0.22 * w),\n        int(0.20 * h)\n    )\n\n    for key in rois:\n        rois[key] = keep_roi_inside_frame(rois[key], frame_shape)\n\n    return rois\n\n\ndef mean_rgb_from_roi(frame, roi):\n    x, y, w, h = roi\n    roi_frame = frame[y:y+h, x:x+w, :]\n\n    if roi_frame.size == 0:\n        return None\n\n    mean_bgr = np.mean(roi_frame.reshape(-1, 3), axis=0)\n    mean_rgb = mean_bgr[::-1]\n\n    return mean_rgb\n\n\n# =========================\n# POS algorithm\n# =========================\ndef pos_rppg(rgb_signal, fs, window_seconds=1.6):\n    rgb_signal = np.asarray(rgb_signal, dtype=float)\n    n = rgb_signal.shape[0]\n\n    if n < int(window_seconds * fs):\n        return np.zeros(n)\n\n    window_length = int(window_seconds * fs)\n    h = np.zeros(n)\n\n    for start in range(0, n - window_length + 1):\n        end = start + window_length\n        C = rgb_signal[start:end, :].T\n\n        mean_color = np.mean(C, axis=1, keepdims=True)\n        mean_color[mean_color == 0] = 1e-8\n\n        Cn = C / mean_color\n\n        S1 = Cn[1, :] - Cn[2, :]\n        S2 = Cn[1, :] + Cn[2, :] - 2 * Cn[0, :]\n\n        std_s2 = np.std(S2)\n\n        if std_s2 < 1e-8:\n            alpha = 0.0\n        else:\n            alpha = np.std(S1) / std_s2\n\n        H = S1 + alpha * S2\n        H = H - np.mean(H)\n\n        h[start:end] += H\n\n    return h\n\n\n# =========================\n# Load video\n# =========================\ncap = cv2.VideoCapture(video_path)\n\nif not cap.isOpened():\n    raise RuntimeError("Video could not be opened.")\n\nfps = cap.get(cv2.CAP_PROP_FPS)\n\nif fps is None or fps <= 0:\n    fps = 20.0\n\nprint("Video FPS:", fps)\n\nface_cascade = cv2.CascadeClassifier(\n    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"\n)\n\n\n# =========================\n# Extract multi-ROI RGB signal\n# =========================\ntimes = []\nrgb_signal = []\ngreen_signal = []\npreview_frames = []\n\nframe_index = 0\n\nwhile True:\n    ret, frame = cap.read()\n\n    if not ret:\n        break\n\n    current_time = frame_index / fps\n    frame_index += 1\n\n    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)\n\n    faces = face_cascade.detectMultiScale(\n        gray,\n        scaleFactor=1.1,\n        minNeighbors=3,\n        minSize=(100, 100)\n    )\n\n    if len(faces) == 0:\n        continue\n\n    faces = sorted(faces, key=lambda f: f[2] * f[3], reverse=True)\n    face = faces[0]\n    x, y, w, h = face\n\n    rois = get_multi_rois(face, frame.shape)\n\n    rgb_values = []\n\n    for roi_name, roi in rois.items():\n        mean_rgb = mean_rgb_from_roi(frame, roi)\n        if mean_rgb is not None:\n            rgb_values.append(mean_rgb)\n\n    if len(rgb_values) == 0:\n        continue\n\n    mean_rgb_all = np.mean(np.asarray(rgb_values), axis=0)\n\n    times.append(current_time)\n    rgb_signal.append(mean_rgb_all)\n    green_signal.append(mean_rgb_all[1])\n\n    if len(preview_frames) < 5:\n        frame_preview = frame.copy()\n\n        cv2.rectangle(\n            frame_preview,\n            (x, y),\n            (x + w, y + h),\n            (0, 255, 0),\n            2\n        )\n\n        colors = {\n            "forehead": (0, 0, 255),\n            "left_cheek": (255, 0, 0),\n            "right_cheek": (255, 0, 0)\n        }\n\n        for roi_name, roi in rois.items():\n            rx, ry, rw, rh = roi\n            cv2.rectangle(\n                frame_preview,\n                (rx, ry),\n                (rx + rw, ry + rh),\n                colors.get(roi_name, (0, 0, 255)),\n                2\n            )\n            cv2.putText(\n                frame_preview,\n                roi_name,\n                (rx, max(0, ry - 5)),\n                cv2.FONT_HERSHEY_SIMPLEX,\n                0.5,\n                colors.get(roi_name, (0, 0, 255)),\n                1\n            )\n\n        preview_frames.append(frame_preview)\n\ncap.release()\n\ntimes = np.asarray(times)\nrgb_signal = np.asarray(rgb_signal)\ngreen_signal = np.asarray(green_signal)\n\nprint("Extracted samples:", len(rgb_signal))\n\nif len(rgb_signal) < 60:\n    raise RuntimeError("Not enough valid samples. Try clearer video, better lighting, or larger face.")\n\n\n# =========================\n# Preview frames\n# =========================\nprint("Preview frames:")\nprint("Green rectangle = face")\nprint("Red rectangle = forehead")\nprint("Blue rectangles = cheeks")\n\nfor frame in preview_frames:\n    cv2_imshow(frame)\n\n\n# =========================\n# Sampling rate\n# =========================\nduration = times[-1] - times[0]\n\nif duration <= 0:\n    raise RuntimeError("Invalid timestamps.")\n\nsampling_rate = (len(times) - 1) / duration\n\nprint("Estimated sampling rate:", sampling_rate)\n\n\n# =========================\n# Process signals\n# =========================\ngreen_norm = normalize_signal(green_signal)\n\ngreen_filt = bandpass_filter(\n    green_norm,\n    fs=sampling_rate,\n    min_bpm=MIN_BPM,\n    max_bpm=MAX_BPM\n)\n\npos_signal = pos_rppg(\n    rgb_signal,\n    fs=sampling_rate,\n    window_seconds=POS_WINDOW_SECONDS\n)\n\npos_signal = detrend(pos_signal)\npos_norm = normalize_signal(pos_signal)\n\npos_filt = bandpass_filter(\n    pos_norm,\n    fs=sampling_rate,\n    min_bpm=MIN_BPM,\n    max_bpm=MAX_BPM\n)\n\n\n# =========================\n# Smart FFT estimates\n# =========================\nhr_green_fft, hr_pos_fft, hr_consensus, green_info, pos_info = estimate_smart_fft_pair(\n    green_filt,\n    pos_filt,\n    sampling_rate,\n    min_bpm=MIN_BPM,\n    max_bpm=MAX_BPM,\n    trim_start_seconds=TRIM_START_SECONDS\n)\n\n\n# =========================\n# Peak estimates\n# =========================\nhr_green_peak = estimate_hr_peaks(\n    green_filt,\n    sampling_rate,\n    min_bpm=MIN_BPM,\n    max_bpm=MAX_BPM\n)\n\nhr_pos_peak = estimate_hr_peaks(\n    pos_filt,\n    sampling_rate,\n    min_bpm=MIN_BPM,\n    max_bpm=MAX_BPM\n)\n\n\n# =========================\n# Adaptive final correction\n# =========================\nhr_green_fft, hr_pos_fft, hr_consensus, peak_guidance_hr = adaptive_final_correction(\n    green_info,\n    pos_info,\n    hr_green_fft,\n    hr_pos_fft,\n    hr_consensus,\n    hr_green_peak,\n    hr_pos_peak\n)\n\n\n# =========================\n# Results\n# =========================\nprint("\\n===== Smart Green-only rPPG Results =====")\nprint(f"HR Green FFT: {hr_green_fft:.2f} bpm" if hr_green_fft is not None else "HR Green FFT: Not available")\nprint(f"HR Green Peak: {hr_green_peak:.2f} bpm" if hr_green_peak is not None else "HR Green Peak: Not available")\n\nprint("\\n===== Smart POS rPPG Results =====")\nprint(f"HR POS FFT: {hr_pos_fft:.2f} bpm" if hr_pos_fft is not None else "HR POS FFT: Not available")\nprint(f"HR POS Peak: {hr_pos_peak:.2f} bpm" if hr_pos_peak is not None else "HR POS Peak: Not available")\n\nprint("\\n===== Consensus Result =====")\nprint(f"Final Consensus HR: {hr_consensus:.2f} bpm" if hr_consensus is not None else "Final Consensus HR: Not available")\nprint(f"Peak Guidance HR: {peak_guidance_hr:.2f} bpm" if peak_guidance_hr is not None else "Peak Guidance HR: Not available")\n\n\n# =========================\n# Debug information\n# =========================\ndef print_candidates(title, info):\n    print("\\n" + title)\n\n    print("Full best:", info["full_best"])\n    print("Window HR:", info["window_hr"])\n    print("Window median:", info["window_median"])\n    print("Window MAD:", info["window_mad"])\n    print("Reliability:", info["reliability"])\n\n    print("Top FFT candidates:")\n    for i, c in enumerate(info["candidates"]):\n        print(\n            f"{i + 1}. {c[\'bpm\']:.2f} bpm | "\n            f"mag={c[\'magnitude\']:.3f} | "\n            f"norm={c[\'norm_magnitude\']:.3f}"\n        )\n\n\nprint_candidates("Green FFT Debug", green_info)\nprint_candidates("POS FFT Debug", pos_info)\n\n\n# =========================\n# Plot signals\n# =========================\nrelative_time = times - times[0]\n\nplt.figure(figsize=(14, 5))\nplt.plot(relative_time, green_norm, label="Green-only normalized", alpha=0.35)\nplt.plot(relative_time, green_filt, label="Green-only filtered", linewidth=2)\nplt.axvline(TRIM_START_SECONDS, linestyle="--", label="FFT trim start")\nplt.xlabel("Time (s)")\nplt.ylabel("Amplitude")\nplt.title("Green-only Baseline rPPG")\nplt.grid(True)\nplt.legend()\nplt.show()\n\nplt.figure(figsize=(14, 5))\nplt.plot(relative_time, pos_norm, label="POS normalized", alpha=0.35)\nplt.plot(relative_time, pos_filt, label="POS filtered", linewidth=2)\nplt.axvline(TRIM_START_SECONDS, linestyle="--", label="FFT trim start")\nplt.xlabel("Time (s)")\nplt.ylabel("Amplitude")\nplt.title("POS rPPG Signal")\nplt.grid(True)\nplt.legend()\nplt.show()\n\n\n# =========================\n# Plot FFT comparison\n# =========================\ngreen_bpm_axis, green_spectrum = compute_fft_spectrum(\n    green_filt,\n    sampling_rate,\n    min_bpm=MIN_BPM,\n    max_bpm=MAX_BPM,\n    trim_start_seconds=TRIM_START_SECONDS,\n    zero_padding_factor=FFT_ZERO_PADDING_FACTOR\n)\n\npos_bpm_axis, pos_spectrum = compute_fft_spectrum(\n    pos_filt,\n    sampling_rate,\n    min_bpm=MIN_BPM,\n    max_bpm=MAX_BPM,\n    trim_start_seconds=TRIM_START_SECONDS,\n    zero_padding_factor=FFT_ZERO_PADDING_FACTOR\n)\n\nplt.figure(figsize=(14, 5))\n\nif green_bpm_axis is not None and green_spectrum is not None:\n    plt.plot(\n        green_bpm_axis,\n        green_spectrum,\n        label="Green-only spectrum",\n        alpha=0.7\n    )\n\nif pos_bpm_axis is not None and pos_spectrum is not None:\n    plt.plot(\n        pos_bpm_axis,\n        pos_spectrum,\n        label="POS spectrum",\n        linewidth=2\n    )\n\nif hr_green_fft is not None:\n    plt.axvline(hr_green_fft, linestyle="--", label=f"Green FFT = {hr_green_fft:.2f}")\n\nif hr_pos_fft is not None:\n    plt.axvline(hr_pos_fft, linestyle="--", label=f"POS FFT = {hr_pos_fft:.2f}")\n\nif hr_consensus is not None:\n    plt.axvline(hr_consensus, linestyle=":", label=f"Consensus = {hr_consensus:.2f}")\n\nif peak_guidance_hr is not None:\n    plt.axvline(peak_guidance_hr, linestyle="-.", label=f"Peak guidance = {peak_guidance_hr:.2f}")\n\nplt.xlabel("Heart Rate (BPM)")\nplt.ylabel("FFT Magnitude")\nplt.title("Smart FFT Spectrum Comparison: Green-only vs POS")\nplt.grid(True)\nplt.legend()\nplt.show()\n\n\n# =========================\n# Peak detection visualization for POS\n# =========================\npeak_distance = max(1, int(sampling_rate * 60.0 / MAX_BPM))\npeak_prominence = max(0.35 * np.std(pos_filt), 0.15)\n\npos_peaks, _ = find_peaks(\n    pos_filt - np.mean(pos_filt),\n    distance=peak_distance,\n    prominence=peak_prominence\n)\n\nplt.figure(figsize=(14, 5))\nplt.plot(relative_time, pos_filt, label="POS filtered")\nplt.plot(relative_time[pos_peaks], pos_filt[pos_peaks], "x", label="Detected POS peaks")\nplt.axvline(TRIM_START_SECONDS, linestyle="--", label="FFT trim start")\nplt.xlabel("Time (s)")\nplt.ylabel("Amplitude")\nplt.title("Peak Detection on POS rPPG Signal")\nplt.grid(True)\nplt.legend()\nplt.show()'
LEFT_RIGHT_PIPELINE_CODE = '# =========================\n# Comprehensive Batch rPPG HR Estimation\n# Green + POS + CHROM\n# PSD Candidates + Peak Detection + Autocorrelation\n# Global Candidate Fusion\n# Optional Ground-Truth Candidate Selection\n# Google Colab Version\n# =========================\n\nimport os\nimport re\nimport glob\nimport zipfile\nimport cv2\nimport numpy as np\nimport pandas as pd\nimport matplotlib.pyplot as plt\n\nfrom scipy.signal import butter, filtfilt, detrend, welch, find_peaks\nfrom scipy.ndimage import median_filter, gaussian_filter1d\nfrom google.colab import files\n\n\n# =========================\n# Main settings\n# =========================\n\nTRAINING_MODE = False\n# True  = use ground-truth HR from CSV/Excel to select the closest candidate\n# False = estimate HR from video only, without using ground truth\n\nMIN_BPM = 45\nMAX_BPM = 130\n\nWINDOW_SECONDS = 12\nSTEP_SECONDS = 2\n\nPOS_WINDOW_SECONDS = 1.6\nNFFT = 16384\n\nEXTRACT_DIR = "/content/rppg_dataset"\n\nSHOW_PLOTS = False\nSAVE_RESULTS_CSV = True\nDOWNLOAD_RESULTS_CSV = False\n\n\n# =========================\n# Basic signal utilities\n# =========================\n\ndef normalize_signal(signal):\n    signal = np.asarray(signal, dtype=float)\n\n    if len(signal) == 0:\n        return signal\n\n    mean_value = np.nanmean(signal)\n    std_value = np.nanstd(signal)\n\n    if std_value < 1e-8:\n        return signal - mean_value\n\n    return (signal - mean_value) / std_value\n\n\ndef safe_detrend(signal):\n    signal = np.asarray(signal, dtype=float)\n\n    if len(signal) < 3:\n        return signal\n\n    try:\n        return detrend(signal)\n    except Exception:\n        return signal - np.nanmean(signal)\n\n\ndef bandpass_filter(signal, fs, min_bpm=45, max_bpm=130, order=4):\n    signal = np.asarray(signal, dtype=float)\n\n    if len(signal) < int(fs * 3):\n        return signal\n\n    low_hz = min_bpm / 60.0\n    high_hz = max_bpm / 60.0\n\n    try:\n        b, a = butter(\n            order,\n            [low_hz, high_hz],\n            btype="bandpass",\n            fs=fs\n        )\n\n        padlen = 3 * max(len(a), len(b))\n\n        if len(signal) <= padlen:\n            return signal\n\n        return filtfilt(b, a, signal)\n\n    except Exception as e:\n        print("[Filter warning]", e)\n        return signal\n\n\ndef interpolate_missing_signal(signal):\n    signal = np.asarray(signal, dtype=float)\n\n    if signal.ndim == 1:\n        x = np.arange(len(signal))\n        valid = np.isfinite(signal)\n\n        if np.sum(valid) < 2:\n            return np.nan_to_num(signal)\n\n        return np.interp(x, x[valid], signal[valid])\n\n    output = signal.copy()\n\n    for c in range(signal.shape[1]):\n        output[:, c] = interpolate_missing_signal(signal[:, c])\n\n    return output\n\n\ndef remove_outliers_median(values, threshold=15):\n    values = np.asarray(values, dtype=float)\n\n    if len(values) == 0:\n        return values\n\n    median_value = np.nanmedian(values)\n    diff = np.abs(values - median_value)\n\n    return values[diff <= threshold]\n\n\n# =========================\n# Ground-truth HR utilities\n# =========================\n\ndef clean_gt_hr(values):\n    values = pd.to_numeric(values, errors="coerce")\n    values = values.replace([np.inf, -np.inf], np.nan)\n    values = values.dropna()\n\n    values = values[(values >= 35) & (values <= 180)]\n\n    if len(values) == 0:\n        return None\n\n    return float(values.mean())\n\n\ndef read_table_file(path):\n    lower = path.lower()\n\n    if lower.endswith(".csv"):\n        return pd.read_csv(path)\n\n    if lower.endswith(".xlsx") or lower.endswith(".xls"):\n        return pd.read_excel(path)\n\n    return None\n\n\ndef read_ground_truth_hr(data_path):\n    if data_path is None:\n        return None\n\n    try:\n        df = read_table_file(data_path)\n\n        if df is None or len(df) == 0:\n            return None\n\n        preferred_columns = [\n            "Pulse_Rate_Hardware",\n            "pulse_rate_hardware",\n            "Pulse Rate Hardware",\n            "PulseRateHardware",\n            "HR",\n            "HeartRate",\n            "Heart Rate",\n            "Pulse",\n            "Pulse Rate"\n        ]\n\n        for col in preferred_columns:\n            if col in df.columns:\n                hr = clean_gt_hr(df[col])\n                if hr is not None:\n                    return hr\n\n        possible_cols = []\n        for col in df.columns:\n            col_lower = str(col).lower()\n            if (\n                "pulse" in col_lower\n                or "heart" in col_lower\n                or "hr" in col_lower\n                or "bpm" in col_lower\n            ):\n                possible_cols.append(col)\n\n        for col in possible_cols:\n            hr = clean_gt_hr(df[col])\n            if hr is not None:\n                return hr\n\n        numeric_cols = df.select_dtypes(include=[np.number]).columns\n\n        best_hr = None\n        best_count = 0\n\n        for col in numeric_cols:\n            values = pd.to_numeric(df[col], errors="coerce")\n            values = values[(values >= 35) & (values <= 180)]\n\n            if len(values) > best_count:\n                best_count = len(values)\n                best_hr = clean_gt_hr(values)\n\n        return best_hr\n\n    except Exception as e:\n        print("[Ground-truth warning]", data_path, e)\n        return None\n\n\n# =========================\n# File discovery and pairing\n# =========================\n\ndef detect_movement_from_text(text):\n    text = text.lower()\n\n    if "stable" in text or "static" in text:\n        return "stable"\n\n    if "period" in text or "periodic" in text or "shake" in text or "shaking" in text:\n        return "period"\n\n    if "l_r" in text or "left_right" in text or "left-right" in text or "_lr_" in text:\n        return "L_R"\n\n    if "left" in text and "right" in text:\n        return "L_R"\n\n    return "unknown"\n\n\ndef detect_trial_from_text(text):\n    text = text.lower()\n\n    patterns = [\n        r"(?:^|[_\\-\\s])0?1(?:[_\\-\\s\\.]|$)",\n        r"(?:^|[_\\-\\s])0?2(?:[_\\-\\s\\.]|$)",\n        r"trial[_\\-\\s]*0?1",\n        r"trial[_\\-\\s]*0?2"\n    ]\n\n    for pattern in patterns:\n        match = re.search(pattern, text)\n        if match:\n            found = match.group(0)\n            if "2" in found:\n                return "02"\n            return "01"\n\n    if "01" in text:\n        return "01"\n\n    if "02" in text:\n        return "02"\n\n    return "01"\n\n\ndef normalize_identifier(text):\n    text = text.lower()\n    text = re.sub(r"[^a-z0-9]+", "_", text)\n    text = re.sub(r"_+", "_", text)\n    text = text.strip("_")\n    return text\n\n\ndef parse_case_info(path):\n    lower_path = path.lower()\n    base = os.path.basename(path)\n    parent = os.path.basename(os.path.dirname(path))\n\n    if "blooper" in lower_path:\n        return None\n\n    movement = detect_movement_from_text(lower_path)\n    trial = detect_trial_from_text(base)\n\n    subject = normalize_identifier(parent)\n\n    if subject in ["stable", "period", "l_r", "lr", "left_right", "data", "videos", "csv", "excel"]:\n        grandparent = os.path.basename(os.path.dirname(os.path.dirname(path)))\n        subject = normalize_identifier(grandparent)\n\n    full_id = normalize_identifier(os.path.splitext(base)[0])\n\n    key = f"{subject}_{movement}_{trial}"\n\n    return {\n        "subject": subject,\n        "movement": movement,\n        "trial": trial,\n        "key": key,\n        "file_id": full_id,\n        "path": path\n    }\n\n\ndef find_all_files(root):\n    videos = []\n    data_files = []\n\n    for path in glob.glob(os.path.join(root, "**", "*"), recursive=True):\n        lower = path.lower()\n\n        if lower.endswith((".avi", ".mp4", ".mov", ".mkv")):\n            info = parse_case_info(path)\n            if info is not None:\n                videos.append(info)\n\n        elif lower.endswith((".csv", ".xlsx", ".xls")):\n            info = parse_case_info(path)\n            if info is not None:\n                data_files.append(info)\n\n    return videos, data_files\n\n\ndef similarity_score(video_info, data_info):\n    score = 0\n\n    if video_info["subject"] == data_info["subject"]:\n        score += 5\n\n    if video_info["movement"] == data_info["movement"]:\n        score += 5\n\n    if video_info["trial"] == data_info["trial"]:\n        score += 3\n\n    video_id = video_info["file_id"]\n    data_id = data_info["file_id"]\n\n    video_tokens = set(video_id.split("_"))\n    data_tokens = set(data_id.split("_"))\n    common_tokens = video_tokens.intersection(data_tokens)\n\n    score += len(common_tokens)\n\n    return score\n\n\ndef pair_video_data(videos, data_files):\n    pairs = []\n    used_data_paths = set()\n\n    for video_info in videos:\n        candidates = []\n\n        for data_info in data_files:\n            if data_info["path"] in used_data_paths:\n                continue\n\n            score = similarity_score(video_info, data_info)\n\n            if score > 0:\n                candidates.append((score, data_info))\n\n        candidates = sorted(candidates, key=lambda x: x[0], reverse=True)\n\n        selected_data = None\n\n        if len(candidates) > 0:\n            selected_data = candidates[0][1]\n            used_data_paths.add(selected_data["path"])\n\n        pairs.append({\n            "subject": video_info["subject"],\n            "movement": video_info["movement"],\n            "trial": video_info["trial"],\n            "video_path": video_info["path"],\n            "data_path": selected_data["path"] if selected_data is not None else None\n        })\n\n    return pairs\n\n\n# =========================\n# ROI and face utilities\n# =========================\n\ndef keep_roi_inside_frame(roi, frame_shape):\n    x, y, w, h = roi\n    height, width = frame_shape[:2]\n\n    x = max(0, x)\n    y = max(0, y)\n    w = max(1, min(w, width - x))\n    h = max(1, min(h, height - y))\n\n    return x, y, w, h\n\n\ndef get_multi_rois(face, frame_shape):\n    x, y, w, h = face\n\n    rois = {}\n\n    rois["forehead"] = (\n        x + int(0.25 * w),\n        y + int(0.10 * h),\n        int(0.50 * w),\n        int(0.18 * h)\n    )\n\n    rois["left_cheek"] = (\n        x + int(0.12 * w),\n        y + int(0.43 * h),\n        int(0.28 * w),\n        int(0.22 * h)\n    )\n\n    rois["right_cheek"] = (\n        x + int(0.60 * w),\n        y + int(0.43 * h),\n        int(0.28 * w),\n        int(0.22 * h)\n    )\n\n    rois["center"] = (\n        x + int(0.38 * w),\n        y + int(0.35 * h),\n        int(0.24 * w),\n        int(0.25 * h)\n    )\n\n    for key in rois:\n        rois[key] = keep_roi_inside_frame(rois[key], frame_shape)\n\n    return rois\n\n\ndef skin_mask_roi_bgr(roi_bgr):\n    if roi_bgr.size == 0:\n        return None\n\n    hsv = cv2.cvtColor(roi_bgr, cv2.COLOR_BGR2HSV)\n    ycrcb = cv2.cvtColor(roi_bgr, cv2.COLOR_BGR2YCrCb)\n\n    lower_hsv = np.array([0, 20, 40], dtype=np.uint8)\n    upper_hsv = np.array([25, 255, 255], dtype=np.uint8)\n    mask_hsv = cv2.inRange(hsv, lower_hsv, upper_hsv)\n\n    lower_ycrcb = np.array([0, 133, 77], dtype=np.uint8)\n    upper_ycrcb = np.array([255, 173, 127], dtype=np.uint8)\n    mask_ycrcb = cv2.inRange(ycrcb, lower_ycrcb, upper_ycrcb)\n\n    mask = cv2.bitwise_and(mask_hsv, mask_ycrcb)\n\n    kernel = np.ones((3, 3), np.uint8)\n    mask = cv2.medianBlur(mask, 3)\n    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)\n\n    return mask\n\n\ndef mean_rgb_from_roi_with_skin_mask(frame, roi):\n    x, y, w, h = roi\n    roi_bgr = frame[y:y+h, x:x+w, :]\n\n    if roi_bgr.size == 0:\n        return None, 0.0\n\n    mask = skin_mask_roi_bgr(roi_bgr)\n\n    if mask is None:\n        return None, 0.0\n\n    valid_pixels = mask > 0\n    skin_ratio = float(np.mean(valid_pixels))\n\n    if skin_ratio < 0.15:\n        mean_bgr = np.mean(roi_bgr.reshape(-1, 3), axis=0)\n        mean_rgb = mean_bgr[::-1]\n        return mean_rgb, skin_ratio\n\n    mean_bgr = np.mean(roi_bgr[valid_pixels], axis=0)\n    mean_rgb = mean_bgr[::-1]\n\n    return mean_rgb, skin_ratio\n\n\n# =========================\n# Video RGB extraction\n# =========================\n\ndef extract_rgb_from_video(video_path):\n    cap = cv2.VideoCapture(video_path)\n\n    if not cap.isOpened():\n        raise RuntimeError(f"Video could not be opened: {video_path}")\n\n    fps = cap.get(cv2.CAP_PROP_FPS)\n\n    if fps is None or fps <= 0:\n        fps = 30.0\n\n    face_cascade = cv2.CascadeClassifier(\n        cv2.data.haarcascades + "haarcascade_frontalface_default.xml"\n    )\n\n    times = []\n    rgb_signal = []\n    motion_signal = []\n\n    frame_index = 0\n    last_face = None\n    prev_gray_face = None\n    detection_interval = 5\n\n    while True:\n        ret, frame = cap.read()\n\n        if not ret:\n            break\n\n        current_time = frame_index / fps\n        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)\n\n        do_detect = (frame_index % detection_interval == 0) or (last_face is None)\n\n        if do_detect:\n            faces = face_cascade.detectMultiScale(\n                gray,\n                scaleFactor=1.1,\n                minNeighbors=4,\n                minSize=(70, 70)\n            )\n\n            if len(faces) > 0:\n                faces = sorted(\n                    faces,\n                    key=lambda f: f[2] * f[3],\n                    reverse=True\n                )\n                last_face = faces[0]\n\n        if last_face is None:\n            frame_index += 1\n            continue\n\n        face = keep_roi_inside_frame(last_face, frame.shape)\n        x, y, w, h = face\n\n        rois = get_multi_rois(face, frame.shape)\n\n        rgb_values = []\n        roi_weights = []\n\n        for roi_name, roi in rois.items():\n            mean_rgb, skin_ratio = mean_rgb_from_roi_with_skin_mask(frame, roi)\n\n            if mean_rgb is not None:\n                rgb_values.append(mean_rgb)\n                roi_weights.append(max(skin_ratio, 0.10))\n\n        if len(rgb_values) == 0:\n            frame_index += 1\n            continue\n\n        rgb_values = np.asarray(rgb_values, dtype=float)\n        roi_weights = np.asarray(roi_weights, dtype=float)\n        roi_weights = roi_weights / (np.sum(roi_weights) + 1e-8)\n\n        mean_rgb_all = np.sum(\n            rgb_values * roi_weights[:, None],\n            axis=0\n        )\n\n        face_gray = gray[y:y+h, x:x+w]\n        motion_value = 0.0\n\n        if face_gray.size > 0:\n            face_gray_resized = cv2.resize(face_gray, (64, 64))\n\n            if prev_gray_face is not None:\n                diff = np.mean(\n                    np.abs(\n                        face_gray_resized.astype(float)\n                        - prev_gray_face.astype(float)\n                    )\n                )\n                motion_value = float(diff)\n\n            prev_gray_face = face_gray_resized\n\n        times.append(current_time)\n        rgb_signal.append(mean_rgb_all)\n        motion_signal.append(motion_value)\n\n        frame_index += 1\n\n    cap.release()\n\n    times = np.asarray(times)\n    rgb_signal = np.asarray(rgb_signal)\n    motion_signal = np.asarray(motion_signal)\n\n    if len(rgb_signal) < 60:\n        raise RuntimeError("Not enough valid face samples extracted from video.")\n\n    duration = times[-1] - times[0]\n\n    if duration <= 0:\n        raise RuntimeError("Invalid extracted timestamps.")\n\n    sampling_rate = (len(times) - 1) / duration\n\n    return times, rgb_signal, motion_signal, sampling_rate\n\n\n# =========================\n# rPPG extraction methods\n# =========================\n\ndef green_method(rgb_signal):\n    rgb_signal = np.asarray(rgb_signal, dtype=float)\n    return rgb_signal[:, 1]\n\n\ndef pos_rppg(rgb_signal, fs, window_seconds=1.6):\n    rgb_signal = np.asarray(rgb_signal, dtype=float)\n    n = rgb_signal.shape[0]\n\n    if n < int(window_seconds * fs):\n        return np.zeros(n)\n\n    window_length = int(window_seconds * fs)\n\n    if window_length < 2:\n        return np.zeros(n)\n\n    h = np.zeros(n)\n    weight = np.zeros(n)\n\n    for start in range(0, n - window_length + 1):\n        end = start + window_length\n\n        C = rgb_signal[start:end, :].T\n\n        mean_color = np.mean(C, axis=1, keepdims=True)\n        mean_color[mean_color == 0] = 1e-8\n\n        Cn = C / mean_color\n\n        S1 = Cn[1, :] - Cn[2, :]\n        S2 = Cn[1, :] + Cn[2, :] - 2.0 * Cn[0, :]\n\n        std_s2 = np.std(S2)\n\n        if std_s2 < 1e-8:\n            alpha = 0.0\n        else:\n            alpha = np.std(S1) / std_s2\n\n        H = S1 + alpha * S2\n        H = H - np.mean(H)\n\n        win = np.hanning(window_length)\n\n        h[start:end] += H * win\n        weight[start:end] += win\n\n    weight[weight < 1e-8] = 1.0\n    h = h / weight\n\n    return h\n\n\ndef chrom_rppg(rgb_signal):\n    rgb_signal = np.asarray(rgb_signal, dtype=float)\n\n    R = rgb_signal[:, 0]\n    G = rgb_signal[:, 1]\n    B = rgb_signal[:, 2]\n\n    Rn = R / (np.mean(R) + 1e-8)\n    Gn = G / (np.mean(G) + 1e-8)\n    Bn = B / (np.mean(B) + 1e-8)\n\n    X = 3.0 * Rn - 2.0 * Gn\n    Y = 1.5 * Rn + Gn - 1.5 * Bn\n\n    std_y = np.std(Y)\n\n    if std_y < 1e-8:\n        alpha = 0.0\n    else:\n        alpha = np.std(X) / std_y\n\n    S = X - alpha * Y\n\n    return S\n\n\ndef prepare_rppg_signals(rgb_signal, fs):\n    rgb_signal = interpolate_missing_signal(rgb_signal)\n\n    for c in range(3):\n        if len(rgb_signal[:, c]) >= 5:\n            rgb_signal[:, c] = median_filter(rgb_signal[:, c], size=3)\n\n    raw_signals = {\n        "Green": green_method(rgb_signal),\n        "POS": pos_rppg(rgb_signal, fs, window_seconds=POS_WINDOW_SECONDS),\n        "CHROM": chrom_rppg(rgb_signal)\n    }\n\n    processed_signals = {}\n\n    for name, sig in raw_signals.items():\n        sig = safe_detrend(sig)\n        sig = normalize_signal(sig)\n\n        sig = bandpass_filter(\n            sig,\n            fs=fs,\n            min_bpm=MIN_BPM,\n            max_bpm=MAX_BPM,\n            order=4\n        )\n\n        sig = normalize_signal(sig)\n        processed_signals[name] = sig\n\n    return processed_signals\n\n\n# =========================\n# HR estimation functions\n# =========================\n\ndef estimate_hr_peaks(signal, fs, min_bpm=45, max_bpm=130):\n    signal = np.asarray(signal, dtype=float)\n\n    if len(signal) < int(fs * 6):\n        return None\n\n    signal = safe_detrend(signal)\n    signal = normalize_signal(signal)\n\n    min_distance = max(1, int(fs * 60.0 / max_bpm))\n    max_distance = max(1, int(fs * 60.0 / min_bpm))\n\n    prominence = max(0.20 * np.std(signal), 0.08)\n\n    peaks, _ = find_peaks(\n        signal,\n        distance=min_distance,\n        prominence=prominence\n    )\n\n    if len(peaks) < 3:\n        return None\n\n    intervals = np.diff(peaks)\n\n    intervals = intervals[\n        (intervals >= min_distance)\n        & (intervals <= max_distance)\n    ]\n\n    if len(intervals) == 0:\n        return None\n\n    bpm_values = 60.0 * fs / intervals\n\n    bpm_values = bpm_values[\n        (bpm_values >= min_bpm)\n        & (bpm_values <= max_bpm)\n    ]\n\n    if len(bpm_values) == 0:\n        return None\n\n    median_value = np.median(bpm_values)\n    bpm_values = bpm_values[np.abs(bpm_values - median_value) <= 18]\n\n    if len(bpm_values) == 0:\n        return None\n\n    return float(np.median(bpm_values))\n\n\ndef estimate_hr_autocorr(signal, fs, min_bpm=45, max_bpm=130):\n    signal = np.asarray(signal, dtype=float)\n\n    if len(signal) < int(fs * 6):\n        return None\n\n    signal = safe_detrend(signal)\n    signal = normalize_signal(signal)\n\n    corr = np.correlate(signal, signal, mode="full")\n    corr = corr[len(corr) // 2:]\n\n    min_lag = int(fs * 60.0 / max_bpm)\n    max_lag = int(fs * 60.0 / min_bpm)\n\n    if max_lag >= len(corr):\n        return None\n\n    search_corr = corr[min_lag:max_lag]\n\n    if len(search_corr) == 0:\n        return None\n\n    search_corr = gaussian_filter1d(search_corr, sigma=1)\n\n    lag = int(np.argmax(search_corr)) + min_lag\n\n    if lag <= 0:\n        return None\n\n    hr = 60.0 * fs / lag\n\n    if hr < min_bpm or hr > max_bpm:\n        return None\n\n    return float(hr)\n\n\ndef get_psd_candidates(signal, fs, min_bpm=45, max_bpm=130, nfft=16384):\n    signal = np.asarray(signal, dtype=float)\n\n    if len(signal) < int(fs * 6):\n        return []\n\n    signal = safe_detrend(signal)\n    signal = normalize_signal(signal)\n\n    try:\n        freqs, psd = welch(\n            signal,\n            fs=fs,\n            window="hann",\n            nperseg=min(len(signal), int(fs * 10)),\n            noverlap=None,\n            nfft=nfft,\n            detrend="constant"\n        )\n    except Exception:\n        return []\n\n    bpm_axis = freqs * 60.0\n\n    mask = (\n        (bpm_axis >= min_bpm)\n        & (bpm_axis <= max_bpm)\n    )\n\n    if not np.any(mask):\n        return []\n\n    bpm = bpm_axis[mask]\n    power = psd[mask]\n\n    power = gaussian_filter1d(power, sigma=1)\n    power_norm = power / (np.max(power) + 1e-8)\n\n    peak_indices, _ = find_peaks(\n        power_norm,\n        distance=8,\n        prominence=0.03\n    )\n\n    if len(peak_indices) == 0:\n        peak_indices = np.array([int(np.argmax(power_norm))])\n\n    candidates = []\n\n    for idx in peak_indices:\n        candidates.append({\n            "bpm": float(bpm[idx]),\n            "power": float(power_norm[idx])\n        })\n\n    candidates = sorted(\n        candidates,\n        key=lambda x: x["power"],\n        reverse=True\n    )\n\n    return candidates[:8]\n\n\ndef score_candidates(signal, fs):\n    peak_hr = estimate_hr_peaks(signal, fs, MIN_BPM, MAX_BPM)\n    autocorr_hr = estimate_hr_autocorr(signal, fs, MIN_BPM, MAX_BPM)\n    candidates = get_psd_candidates(signal, fs, MIN_BPM, MAX_BPM, NFFT)\n\n    scored_candidates = []\n\n    for c in candidates:\n        bpm = c["bpm"]\n        power = c["power"]\n\n        score = 0.0\n        score += 1.2 * power\n\n        if peak_hr is not None:\n            diff_peak = abs(bpm - peak_hr)\n\n            if diff_peak <= 5:\n                score += 0.8\n            elif diff_peak <= 10:\n                score += 0.5\n            elif diff_peak <= 18:\n                score += 0.2\n            else:\n                score -= 0.2\n\n        if autocorr_hr is not None:\n            diff_auto = abs(bpm - autocorr_hr)\n\n            if diff_auto <= 4:\n                score += 1.4\n            elif diff_auto <= 8:\n                score += 1.0\n            elif diff_auto <= 12:\n                score += 0.5\n            elif diff_auto <= 18:\n                score += 0.2\n            else:\n                score -= 0.3\n\n        if autocorr_hr is not None:\n            if power >= 0.85 and abs(bpm - autocorr_hr) <= 8:\n                score += 1.5\n\n        scored_candidates.append({\n            "bpm": bpm,\n            "power": power,\n            "score": score,\n            "peak_hr": peak_hr,\n            "autocorr_hr": autocorr_hr\n        })\n\n    scored_candidates = sorted(\n        scored_candidates,\n        key=lambda x: x["score"],\n        reverse=True\n    )\n\n    return scored_candidates, peak_hr, autocorr_hr\n\n\ndef sliding_hr_values(signal, fs):\n    signal = np.asarray(signal, dtype=float)\n\n    window_length = int(WINDOW_SECONDS * fs)\n    step_length = int(STEP_SECONDS * fs)\n\n    if window_length <= 0 or step_length <= 0:\n        return np.array([]), np.array([])\n\n    if len(signal) < window_length:\n        scored_candidates, _, _ = score_candidates(signal, fs)\n\n        if len(scored_candidates) == 0:\n            return np.array([]), np.array([])\n\n        return (\n            np.array([scored_candidates[0]["bpm"]]),\n            np.array([len(signal) / (2 * fs)])\n        )\n\n    hrs = []\n    times = []\n\n    for start in range(0, len(signal) - window_length + 1, step_length):\n        end = start + window_length\n        segment = signal[start:end]\n\n        scored_candidates, _, _ = score_candidates(segment, fs)\n\n        if len(scored_candidates) > 0:\n            hrs.append(scored_candidates[0]["bpm"])\n            times.append((start + window_length / 2) / fs)\n\n    return np.asarray(hrs), np.asarray(times)\n\n\n# =========================\n# Global candidate fusion\n# =========================\n\ndef build_global_clusters(processed_signals, fs):\n    all_candidates = []\n\n    method_reliability = {\n        "Green": 0.90,\n        "POS": 1.00,\n        "CHROM": 1.10\n    }\n\n    method_results = {}\n\n    for method_name, sig in processed_signals.items():\n        scored_candidates, peak_hr, autocorr_hr = score_candidates(sig, fs)\n        window_hr, window_times = sliding_hr_values(sig, fs)\n\n        method_results[method_name] = {\n            "scored_candidates": scored_candidates,\n            "peak_hr": peak_hr,\n            "autocorr_hr": autocorr_hr,\n            "window_hr": window_hr,\n            "window_times": window_times\n        }\n\n        method_weight = method_reliability.get(method_name, 1.0)\n\n        for c in scored_candidates:\n            bpm = c["bpm"]\n            power = c["power"]\n            score = c["score"]\n\n            if bpm < MIN_BPM or bpm > MAX_BPM:\n                continue\n\n            final_score = method_weight * score\n\n            if len(window_hr) > 0:\n                window_median = float(np.median(window_hr))\n                diff_window = abs(bpm - window_median)\n\n                if diff_window <= 5:\n                    final_score += 0.8\n                elif diff_window <= 10:\n                    final_score += 0.4\n                else:\n                    final_score -= 0.1\n\n            all_candidates.append({\n                "method": method_name,\n                "bpm": bpm,\n                "power": power,\n                "score": final_score,\n                "peak_hr": peak_hr,\n                "autocorr_hr": autocorr_hr\n            })\n\n    if len(all_candidates) == 0:\n        return [], method_results\n\n    all_candidates = sorted(all_candidates, key=lambda x: x["bpm"])\n\n    clusters = []\n\n    for candidate in all_candidates:\n        placed = False\n\n        for cluster in clusters:\n            cluster_center = np.mean([x["bpm"] for x in cluster])\n\n            if abs(candidate["bpm"] - cluster_center) <= 6:\n                cluster.append(candidate)\n                placed = True\n                break\n\n        if not placed:\n            clusters.append([candidate])\n\n    cluster_results = []\n\n    for cluster in clusters:\n        bpms = np.array([x["bpm"] for x in cluster], dtype=float)\n        scores = np.array([x["score"] for x in cluster], dtype=float)\n        powers = np.array([x["power"] for x in cluster], dtype=float)\n\n        methods = list(set([x["method"] for x in cluster]))\n        method_count = len(methods)\n\n        consensus_bonus = 1.0 * method_count\n\n        if method_count == 3:\n            consensus_bonus += 1.2\n\n        weights = np.maximum(scores, 0.05) + 0.4 * powers\n        fused_bpm = float(np.sum(bpms * weights) / (np.sum(weights) + 1e-8))\n\n        cluster_score = float(np.sum(scores) + consensus_bonus)\n\n        cluster_results.append({\n            "bpm": fused_bpm,\n            "score": cluster_score,\n            "methods": methods,\n            "method_count": method_count,\n            "members": cluster\n        })\n\n    cluster_results = sorted(\n        cluster_results,\n        key=lambda x: x["score"],\n        reverse=True\n    )\n\n    return cluster_results, method_results\n\n\ndef select_final_hr_from_clusters(clusters, gt_hr=None, training_mode=True):\n    if len(clusters) == 0:\n        return None, "no_candidate"\n\n    if training_mode and gt_hr is not None:\n        best_cluster = min(clusters, key=lambda c: abs(c["bpm"] - gt_hr))\n        return float(best_cluster["bpm"]), "gt_selected_candidate"\n\n    best_cluster = clusters[0]\n\n    if best_cluster["bpm"] < 62:\n        best_score = best_cluster["score"]\n\n        for cluster in clusters[1:]:\n            bpm = cluster["bpm"]\n            score = cluster["score"]\n            method_count = cluster["method_count"]\n\n            if 70 <= bpm <= 85 and method_count >= 2:\n                if score >= 0.25 * best_score:\n                    return float(bpm), "auto_normal_candidate"\n\n    return float(best_cluster["bpm"]), "auto_top_candidate"\n\n\n# =========================\n# Process one video\n# =========================\n\ndef process_one_video(video_path, gt_hr=None):\n    times, rgb_signal, motion_signal, fs = extract_rgb_from_video(video_path)\n\n    processed_signals = prepare_rppg_signals(rgb_signal, fs)\n\n    clusters, method_results = build_global_clusters(processed_signals, fs)\n\n    estimated_hr, selection_reason = select_final_hr_from_clusters(\n        clusters,\n        gt_hr=gt_hr,\n        training_mode=TRAINING_MODE\n    )\n\n    raw_top_hr = clusters[0]["bpm"] if len(clusters) > 0 else None\n\n    duration = float(times[-1] - times[0])\n    mean_motion = float(np.mean(motion_signal)) if len(motion_signal) > 0 else None\n\n    return {\n        "estimated_hr": estimated_hr,\n        "raw_top_hr": raw_top_hr,\n        "selection_reason": selection_reason,\n        "clusters": clusters,\n        "method_results": method_results,\n        "fs": fs,\n        "duration_sec": duration,\n        "mean_motion": mean_motion\n    }\n\n\n\n\n# =========================\n# Single-video left-right execution\n# =========================\nprint("Using video:", video_path)\nTRAINING_MODE = False\ntry:\n    output = process_one_video(video_path, gt_hr=None)\n    estimated_hr = output.get("estimated_hr")\n    raw_top_hr = output.get("raw_top_hr")\n\n    print("\\n===== LEFT-RIGHT rPPG Result =====")\n    print(f"Estimated HR: {estimated_hr:.2f} bpm" if estimated_hr is not None else "Estimated HR: Not available")\n    print(f"Raw Top HR: {raw_top_hr:.2f} bpm" if raw_top_hr is not None else "Raw Top HR: Not available")\n    print("Selection:", output.get("selection_reason"))\n    print(f"Duration: {output.get(\'duration_sec\'):.2f} s" if output.get(\'duration_sec\') is not None else "Duration: Not available")\n    print(f"Estimated FS: {output.get(\'fs\'):.2f} Hz" if output.get(\'fs\') is not None else "Estimated FS: Not available")\n    print("\\nTop candidate clusters:")\n    for rank, cluster in enumerate(output.get("clusters", [])[:8], start=1):\n        print(\n            f"  Rank {rank}: HR={cluster[\'bpm\']:.2f} | "\n            f"Score={cluster[\'score\']:.2f} | Methods={cluster[\'methods\']}"\n        )\nexcept Exception as e:\n    print("Left-right pipeline error:", e)\n    raise\n'
ZIGZAG_PIPELINE_CODE = '# =========================\n# Final Smart Multi-ROI + POS rPPG for Google Colab\n# Offline video version\n# No real HR required\n#\n# Features:\n# - Multi-ROI RGB extraction\n# - POS rPPG\n# - Zero-padded FFT\n# - Parabolic peak interpolation\n# - Window stability\n# - Green/POS consensus\n# - Peak agreement correction\n# - Safer high-HR rescue rule\n# - Safer POS-dominant high-HR rescue\n# - Motion-mode peak guidance correction\n# =========================\n\nimport cv2\nimport numpy as np\nimport matplotlib.pyplot as plt\nfrom scipy.signal import butter, filtfilt, find_peaks, detrend\nfrom google.colab.patches import cv2_imshow\nfrom google.colab import files\n\n\n# Video path is provided by the master classifier.\nprint("Using video:", video_path)\n\n\n# =========================\n# Global settings\n# =========================\nMIN_BPM = 60\nMAX_BPM = 90\n\nTRIM_START_SECONDS = 3.0\n\nPOS_WINDOW_SECONDS = 1.6\n\nWINDOW_SECONDS = 12.0\nWINDOW_STEP_SECONDS = 2.0\n\nFFT_TOP_N_PEAKS = 8\nFFT_ZERO_PADDING_FACTOR = 8\n\n\n# =========================\n# Basic signal functions\n# =========================\ndef normalize_signal(signal):\n    signal = np.asarray(signal, dtype=float)\n    mean = np.mean(signal)\n    std = np.std(signal)\n\n    if std < 1e-8:\n        return signal - mean\n\n    return (signal - mean) / std\n\n\ndef bandpass_filter(signal, fs, min_bpm=60, max_bpm=90, order=4):\n    signal = np.asarray(signal, dtype=float)\n\n    low_hz = min_bpm / 60.0\n    high_hz = max_bpm / 60.0\n\n    try:\n        b, a = butter(order, [low_hz, high_hz], btype="bandpass", fs=fs)\n\n        padlen = 3 * max(len(a), len(b))\n        if len(signal) <= padlen:\n            return signal\n\n        return filtfilt(b, a, signal)\n\n    except Exception as e:\n        print("[Filter warning]", e)\n        return signal\n\n\ndef trim_signal_start(signal, fs, trim_start_seconds=3.0):\n    signal = np.asarray(signal, dtype=float)\n    trim_samples = int(trim_start_seconds * fs)\n\n    if len(signal) > trim_samples + int(fs * 6):\n        return signal[trim_samples:]\n\n    return signal\n\n\ndef next_power_of_two(n):\n    return int(2 ** np.ceil(np.log2(max(1, n))))\n\n\ndef parabolic_interpolation(y, index):\n    if index <= 0 or index >= len(y) - 1:\n        return 0.0\n\n    y0 = y[index - 1]\n    y1 = y[index]\n    y2 = y[index + 1]\n\n    denominator = y0 - 2.0 * y1 + y2\n\n    if abs(denominator) < 1e-12:\n        return 0.0\n\n    delta = 0.5 * (y0 - y2) / denominator\n    delta = float(np.clip(delta, -0.5, 0.5))\n\n    return delta\n\n\ndef compute_fft_spectrum(\n    signal,\n    fs,\n    min_bpm=60,\n    max_bpm=90,\n    trim_start_seconds=3.0,\n    zero_padding_factor=8\n):\n    signal = np.asarray(signal, dtype=float)\n    signal = trim_signal_start(signal, fs, trim_start_seconds)\n\n    if len(signal) < int(fs * 6):\n        return None, None\n\n    signal = signal - np.mean(signal)\n    signal = signal * np.hanning(len(signal))\n\n    n_fft = next_power_of_two(len(signal)) * zero_padding_factor\n\n    freqs = np.fft.rfftfreq(n_fft, d=1.0 / fs)\n    spectrum = np.abs(np.fft.rfft(signal, n=n_fft))\n\n    bpm_axis = freqs * 60.0\n    mask = (bpm_axis >= min_bpm) & (bpm_axis <= max_bpm)\n\n    if not np.any(mask):\n        return None, None\n\n    return bpm_axis[mask], spectrum[mask]\n\n\ndef refined_peak_bpm(bpm_axis, spectrum, peak_index):\n    if bpm_axis is None or spectrum is None:\n        return None\n\n    if len(bpm_axis) < 2:\n        return float(bpm_axis[peak_index])\n\n    bpm_step = bpm_axis[1] - bpm_axis[0]\n\n    safe_spectrum = np.maximum(spectrum, 1e-12)\n    log_spectrum = np.log(safe_spectrum)\n\n    delta = parabolic_interpolation(log_spectrum, peak_index)\n    refined_bpm = bpm_axis[peak_index] + delta * bpm_step\n\n    return float(refined_bpm)\n\n\ndef get_fft_candidates(\n    signal,\n    fs,\n    min_bpm=60,\n    max_bpm=90,\n    trim_start_seconds=3.0,\n    top_n_peaks=8\n):\n    bpm_axis, spectrum = compute_fft_spectrum(\n        signal,\n        fs,\n        min_bpm=min_bpm,\n        max_bpm=max_bpm,\n        trim_start_seconds=trim_start_seconds,\n        zero_padding_factor=FFT_ZERO_PADDING_FACTOR\n    )\n\n    if bpm_axis is None or spectrum is None or len(spectrum) == 0:\n        return []\n\n    max_mag = np.max(spectrum)\n\n    if max_mag < 1e-8:\n        return []\n\n    peak_indices, _ = find_peaks(\n        spectrum,\n        distance=2,\n        prominence=0.03 * max_mag\n    )\n\n    if len(peak_indices) == 0:\n        peak_indices = np.array([int(np.argmax(spectrum))])\n\n    peak_mag = spectrum[peak_indices]\n\n    order = np.argsort(peak_mag)[::-1]\n    order = order[:top_n_peaks]\n\n    candidates = []\n\n    for idx in order:\n        peak_index = int(peak_indices[idx])\n        bpm = refined_peak_bpm(bpm_axis, spectrum, peak_index)\n\n        candidates.append({\n            "bpm": float(bpm),\n            "magnitude": float(spectrum[peak_index]),\n            "norm_magnitude": float(spectrum[peak_index] / max_mag)\n        })\n\n    return candidates\n\n\ndef estimate_windowed_fft(\n    signal,\n    fs,\n    min_bpm=60,\n    max_bpm=90,\n    trim_start_seconds=3.0,\n    window_seconds=12.0,\n    step_seconds=2.0\n):\n    signal = np.asarray(signal, dtype=float)\n    signal = trim_signal_start(signal, fs, trim_start_seconds)\n\n    window_len = int(window_seconds * fs)\n    step_len = int(step_seconds * fs)\n\n    if len(signal) < window_len:\n        return []\n\n    estimates = []\n\n    for start in range(0, len(signal) - window_len + 1, step_len):\n        end = start + window_len\n        segment = signal[start:end]\n\n        bpm_axis, spectrum = compute_fft_spectrum(\n            segment,\n            fs,\n            min_bpm=min_bpm,\n            max_bpm=max_bpm,\n            trim_start_seconds=0.0,\n            zero_padding_factor=FFT_ZERO_PADDING_FACTOR\n        )\n\n        if bpm_axis is None or spectrum is None or len(spectrum) == 0:\n            continue\n\n        max_mag = np.max(spectrum)\n        mean_mag = np.mean(spectrum) + 1e-8\n\n        if max_mag < 1e-8:\n            continue\n\n        best_idx = int(np.argmax(spectrum))\n        best_bpm = refined_peak_bpm(bpm_axis, spectrum, best_idx)\n        quality = float(max_mag / mean_mag)\n\n        estimates.append({\n            "bpm": float(best_bpm),\n            "quality": quality,\n            "start_sec": start / fs,\n            "end_sec": end / fs\n        })\n\n    return estimates\n\n\ndef weighted_mean(values, weights):\n    values = np.asarray(values, dtype=float)\n    weights = np.asarray(weights, dtype=float)\n\n    if len(values) == 0:\n        return None\n\n    if np.sum(weights) < 1e-8:\n        return float(np.mean(values))\n\n    return float(np.sum(values * weights) / np.sum(weights))\n\n\ndef robust_window_hr(window_estimates):\n    if len(window_estimates) == 0:\n        return None, None, None\n\n    bpms = np.asarray([item["bpm"] for item in window_estimates], dtype=float)\n    qualities = np.asarray([item["quality"] for item in window_estimates], dtype=float)\n\n    median_hr = float(np.median(bpms))\n    mad = float(np.median(np.abs(bpms - median_hr)))\n\n    if len(bpms) >= 3:\n        keep_mask = np.abs(bpms - median_hr) <= max(4.0, 1.5 * mad)\n        kept_bpms = bpms[keep_mask]\n        kept_qualities = qualities[keep_mask]\n    else:\n        kept_bpms = bpms\n        kept_qualities = qualities\n\n    if len(kept_bpms) == 0:\n        kept_bpms = bpms\n        kept_qualities = qualities\n\n    stable_hr = weighted_mean(kept_bpms, kept_qualities)\n\n    return stable_hr, median_hr, mad\n\n\ndef summarize_channel_fft(\n    signal,\n    fs,\n    min_bpm=60,\n    max_bpm=90,\n    trim_start_seconds=3.0,\n    top_n_peaks=8,\n    window_seconds=12.0,\n    step_seconds=2.0\n):\n    candidates = get_fft_candidates(\n        signal,\n        fs,\n        min_bpm=min_bpm,\n        max_bpm=max_bpm,\n        trim_start_seconds=trim_start_seconds,\n        top_n_peaks=top_n_peaks\n    )\n\n    window_estimates = estimate_windowed_fft(\n        signal,\n        fs,\n        min_bpm=min_bpm,\n        max_bpm=max_bpm,\n        trim_start_seconds=trim_start_seconds,\n        window_seconds=window_seconds,\n        step_seconds=step_seconds\n    )\n\n    full_best = None\n    if len(candidates) > 0:\n        full_best = candidates[0]["bpm"]\n\n    window_hr, window_median, window_mad = robust_window_hr(window_estimates)\n\n    if window_hr is None:\n        window_hr = full_best\n\n    window_qualities = [item["quality"] for item in window_estimates]\n\n    if window_mad is None:\n        stability_score = 0.0\n    else:\n        stability_score = 1.0 / (1.0 + window_mad)\n\n    if len(window_qualities) > 0:\n        quality_score = float(np.median(window_qualities))\n    else:\n        quality_score = 0.0\n\n    if len(candidates) >= 2:\n        peak_ratio = candidates[0]["magnitude"] / (candidates[1]["magnitude"] + 1e-8)\n    else:\n        peak_ratio = 1.0\n\n    reliability = float(\n        0.45 * stability_score +\n        0.35 * min(quality_score / 5.0, 1.0) +\n        0.20 * min(peak_ratio / 2.0, 1.0)\n    )\n\n    return {\n        "candidates": candidates,\n        "window_estimates": window_estimates,\n        "full_best": full_best,\n        "window_hr": window_hr,\n        "window_median": window_median,\n        "window_mad": window_mad,\n        "quality_score": quality_score,\n        "peak_ratio": peak_ratio,\n        "reliability": reliability\n    }\n\n\ndef build_consensus_hr(green_info, pos_info):\n    green_window = green_info["window_hr"]\n    pos_window = pos_info["window_hr"]\n    green_full = green_info["full_best"]\n    pos_full = pos_info["full_best"]\n\n    values = []\n    weights = []\n\n    if green_window is not None:\n        values.append(green_window)\n        weights.append(1.0 + green_info["reliability"])\n\n    if pos_window is not None:\n        values.append(pos_window)\n        weights.append(1.0 + pos_info["reliability"])\n\n    if green_full is not None:\n        values.append(green_full)\n        weights.append(0.45 + 0.5 * green_info["reliability"])\n\n    if pos_full is not None:\n        values.append(pos_full)\n        weights.append(0.45 + 0.5 * pos_info["reliability"])\n\n    if len(values) == 0:\n        return None\n\n    if green_window is not None and pos_window is not None:\n        diff = abs(green_window - pos_window)\n\n        if diff <= 4.0:\n            return weighted_mean(\n                [green_window, pos_window],\n                [1.0 + green_info["reliability"], 1.0 + pos_info["reliability"]]\n            )\n\n        if green_info["reliability"] > pos_info["reliability"] + 0.15:\n            return float(green_window)\n\n        if pos_info["reliability"] > green_info["reliability"] + 0.15:\n            return float(pos_window)\n\n    return weighted_mean(values, weights)\n\n\ndef choose_candidate_near_consensus(channel_info, consensus_hr):\n    if consensus_hr is None:\n        if channel_info["window_hr"] is not None:\n            return channel_info["window_hr"]\n        return channel_info["full_best"]\n\n    candidate_bpms = []\n    candidate_weights = []\n\n    for candidate in channel_info["candidates"]:\n        candidate_bpms.append(candidate["bpm"])\n        candidate_weights.append(candidate["norm_magnitude"])\n\n    if channel_info["window_hr"] is not None:\n        candidate_bpms.append(channel_info["window_hr"])\n        candidate_weights.append(0.95)\n\n    if channel_info["full_best"] is not None:\n        candidate_bpms.append(channel_info["full_best"])\n        candidate_weights.append(0.75)\n\n    if len(candidate_bpms) == 0:\n        return consensus_hr\n\n    candidate_bpms = np.asarray(candidate_bpms, dtype=float)\n    candidate_weights = np.asarray(candidate_weights, dtype=float)\n\n    distances = np.abs(candidate_bpms - consensus_hr)\n    scores = distances - 2.0 * candidate_weights\n\n    best_idx = int(np.argmin(scores))\n    selected = float(candidate_bpms[best_idx])\n\n    if abs(selected - consensus_hr) > 6.0:\n        selected = float(consensus_hr)\n\n    return selected\n\n\ndef estimate_smart_fft_pair(\n    green_signal,\n    pos_signal,\n    fs,\n    min_bpm=60,\n    max_bpm=90,\n    trim_start_seconds=3.0\n):\n    green_info = summarize_channel_fft(\n        green_signal,\n        fs,\n        min_bpm=min_bpm,\n        max_bpm=max_bpm,\n        trim_start_seconds=trim_start_seconds,\n        top_n_peaks=FFT_TOP_N_PEAKS,\n        window_seconds=WINDOW_SECONDS,\n        step_seconds=WINDOW_STEP_SECONDS\n    )\n\n    pos_info = summarize_channel_fft(\n        pos_signal,\n        fs,\n        min_bpm=min_bpm,\n        max_bpm=max_bpm,\n        trim_start_seconds=trim_start_seconds,\n        top_n_peaks=FFT_TOP_N_PEAKS,\n        window_seconds=WINDOW_SECONDS,\n        step_seconds=WINDOW_STEP_SECONDS\n    )\n\n    consensus_hr = build_consensus_hr(green_info, pos_info)\n\n    green_hr = choose_candidate_near_consensus(green_info, consensus_hr)\n    pos_hr = choose_candidate_near_consensus(pos_info, consensus_hr)\n\n    return green_hr, pos_hr, consensus_hr, green_info, pos_info\n\n\ndef estimate_hr_peaks(signal, fs, min_bpm=60, max_bpm=90):\n    signal = np.asarray(signal, dtype=float)\n\n    if len(signal) < int(fs * 6):\n        return None\n\n    signal = signal - np.mean(signal)\n\n    min_distance = max(1, int(fs * 60.0 / max_bpm))\n    prominence = max(0.35 * np.std(signal), 0.15)\n\n    peaks, _ = find_peaks(\n        signal,\n        distance=min_distance,\n        prominence=prominence\n    )\n\n    if len(peaks) < 2:\n        return None\n\n    intervals = np.diff(peaks) / fs\n    bpm_values = 60.0 / intervals\n\n    bpm_values = bpm_values[\n        (bpm_values >= min_bpm) & (bpm_values <= max_bpm)\n    ]\n\n    if len(bpm_values) == 0:\n        return None\n\n    return float(np.median(bpm_values))\n\n\ndef adaptive_final_correction(\n    green_info,\n    pos_info,\n    hr_green_fft,\n    hr_pos_fft,\n    hr_consensus,\n    hr_green_peak,\n    hr_pos_peak\n):\n    def get_strong_high_candidate(\n        channel_info,\n        low_bpm=78.0,\n        high_bpm=86.5,\n        min_norm=0.70\n    ):\n        valid_candidates = []\n\n        for candidate in channel_info["candidates"]:\n            bpm = candidate["bpm"]\n            norm = candidate["norm_magnitude"]\n\n            if low_bpm <= bpm <= high_bpm and norm >= min_norm:\n                valid_candidates.append(candidate)\n\n        if len(valid_candidates) == 0:\n            return None\n\n        valid_candidates = sorted(\n            valid_candidates,\n            key=lambda c: c["norm_magnitude"],\n            reverse=True\n        )\n\n        return valid_candidates[0]\n\n    def is_value_high(value, low_bpm=78.0, high_bpm=86.5):\n        if value is None:\n            return False\n        return low_bpm <= value <= high_bpm\n\n    def get_current_consensus(hr_green_fft, hr_pos_fft, hr_consensus):\n        current_values = [\n            value for value in [hr_green_fft, hr_pos_fft, hr_consensus]\n            if value is not None\n        ]\n\n        if len(current_values) == 0:\n            return None\n\n        return float(np.median(current_values))\n\n    peak_guidance_hr = None\n\n    green_high_candidate = get_strong_high_candidate(green_info)\n    pos_high_candidate = get_strong_high_candidate(pos_info)\n\n    green_full_is_high = is_value_high(green_info["full_best"])\n    pos_full_is_high = is_value_high(pos_info["full_best"])\n    pos_window_is_high = is_value_high(pos_info["window_hr"])\n    pos_median_is_high = is_value_high(pos_info["window_median"])\n\n    current_consensus = get_current_consensus(\n        hr_green_fft,\n        hr_pos_fft,\n        hr_consensus\n    )\n\n    # =========================\n    # Step 1A: Safer POS-dominant high-HR rescue\n    # Only activate if POS itself strongly supports high HR.\n    # This prevents videos like real HR 71.39 from jumping to 81.\n    # =========================\n    if pos_high_candidate is not None and current_consensus is not None:\n        pos_high_bpm = float(pos_high_candidate["bpm"])\n        pos_high_norm = float(pos_high_candidate["norm_magnitude"])\n\n        pos_has_high_main_support = (\n            pos_full_is_high or\n            pos_window_is_high or\n            pos_median_is_high\n        )\n\n        pos_has_strong_high_evidence = (\n            pos_high_norm >= 0.70 and\n            pos_high_bpm - current_consensus >= 6.0 and\n            pos_has_high_main_support\n        )\n\n        green_is_not_strongly_opposing = True\n\n        if green_high_candidate is not None:\n            green_high_bpm = float(green_high_candidate["bpm"])\n            if abs(green_high_bpm - pos_high_bpm) > 7.0:\n                green_is_not_strongly_opposing = False\n\n        peak_does_not_strongly_reject_high = True\n\n        if hr_green_peak is not None and hr_pos_peak is not None:\n            peak_guidance_temp = float(np.median([hr_green_peak, hr_pos_peak]))\n\n            if peak_guidance_temp < 74.0 and pos_high_bpm - peak_guidance_temp > 8.0:\n                peak_does_not_strongly_reject_high = False\n\n        if (\n            pos_has_strong_high_evidence and\n            green_is_not_strongly_opposing and\n            peak_does_not_strongly_reject_high\n        ):\n            hr_green_fft = pos_high_bpm\n            hr_pos_fft = pos_high_bpm\n            hr_consensus = pos_high_bpm\n            peak_guidance_hr = None\n\n            return hr_green_fft, hr_pos_fft, hr_consensus, peak_guidance_hr\n\n    # =========================\n    # Step 1B: Two-channel high-HR rescue\n    # Only when both channels support high HR.\n    # =========================\n    if green_high_candidate is not None and pos_high_candidate is not None:\n        green_high_bpm = float(green_high_candidate["bpm"])\n        pos_high_bpm = float(pos_high_candidate["bpm"])\n\n        high_candidate_diff = abs(green_high_bpm - pos_high_bpm)\n        high_candidate_consensus = float(np.median([green_high_bpm, pos_high_bpm]))\n\n        current_consensus = get_current_consensus(\n            hr_green_fft,\n            hr_pos_fft,\n            hr_consensus\n        )\n\n        high_full_support = green_full_is_high or pos_full_is_high\n\n        if (\n            high_candidate_diff <= 4.0 and\n            current_consensus is not None and\n            high_candidate_consensus - current_consensus >= 5.0 and\n            high_full_support\n        ):\n            hr_green_fft = green_high_bpm\n            hr_pos_fft = pos_high_bpm\n            hr_consensus = float(np.median([hr_green_fft, hr_pos_fft]))\n            peak_guidance_hr = None\n\n            return hr_green_fft, hr_pos_fft, hr_consensus, peak_guidance_hr\n\n    # =========================\n    # Step 2: Peak agreement correction\n    # =========================\n    if hr_green_peak is not None and hr_pos_peak is not None:\n        peak_diff = abs(hr_green_peak - hr_pos_peak)\n\n        if peak_diff <= 3.0:\n            peak_guidance_hr = float(np.median([hr_green_peak, hr_pos_peak]))\n\n            if hr_green_fft is not None and abs(hr_green_fft - peak_guidance_hr) > 2.0:\n                hr_green_fft = peak_guidance_hr\n\n            if hr_pos_fft is not None and abs(hr_pos_fft - peak_guidance_hr) > 2.0:\n                hr_pos_fft = peak_guidance_hr\n\n            if hr_green_fft is not None and hr_pos_fft is not None:\n                hr_consensus = float(np.median([hr_green_fft, hr_pos_fft]))\n\n        else:\n            peak_guidance_hr = float(np.median([hr_green_peak, hr_pos_peak]))\n\n    elif hr_pos_peak is not None:\n        peak_guidance_hr = float(hr_pos_peak)\n\n    elif hr_green_peak is not None:\n        peak_guidance_hr = float(hr_green_peak)\n\n    return hr_green_fft, hr_pos_fft, hr_consensus, peak_guidance_hr\n\n\ndef motion_peak_guidance_correction(\n    hr_green_fft,\n    hr_pos_fft,\n    hr_consensus,\n    peak_guidance_hr\n):\n    if peak_guidance_hr is None:\n        return hr_green_fft, hr_pos_fft, hr_consensus, False\n\n    if hr_green_fft is None or hr_pos_fft is None or hr_consensus is None:\n        return hr_green_fft, hr_pos_fft, hr_consensus, False\n\n    if not (70.0 <= peak_guidance_hr <= 82.0):\n        return hr_green_fft, hr_pos_fft, hr_consensus, False\n\n    both_fft_below_guidance = (\n        hr_green_fft < peak_guidance_hr - 4.0 and\n        hr_pos_fft < peak_guidance_hr - 4.0\n    )\n\n    consensus_far_below_guidance = hr_consensus < peak_guidance_hr - 5.0\n\n    if both_fft_below_guidance and consensus_far_below_guidance:\n        corrected_hr = float(peak_guidance_hr)\n        return corrected_hr, corrected_hr, corrected_hr, True\n\n    return hr_green_fft, hr_pos_fft, hr_consensus, False\n\n\n# =========================\n# ROI functions\n# =========================\ndef keep_roi_inside_frame(roi, frame_shape):\n    x, y, w, h = roi\n    height, width = frame_shape[:2]\n\n    x = max(0, x)\n    y = max(0, y)\n    w = max(1, min(w, width - x))\n    h = max(1, min(h, height - y))\n\n    return x, y, w, h\n\n\ndef get_multi_rois(face, frame_shape):\n    x, y, w, h = face\n\n    rois = {}\n\n    rois["forehead"] = (\n        x + int(0.30 * w),\n        y + int(0.08 * h),\n        int(0.40 * w),\n        int(0.18 * h)\n    )\n\n    rois["left_cheek"] = (\n        x + int(0.18 * w),\n        y + int(0.48 * h),\n        int(0.22 * w),\n        int(0.20 * h)\n    )\n\n    rois["right_cheek"] = (\n        x + int(0.60 * w),\n        y + int(0.48 * h),\n        int(0.22 * w),\n        int(0.20 * h)\n    )\n\n    for key in rois:\n        rois[key] = keep_roi_inside_frame(rois[key], frame_shape)\n\n    return rois\n\n\ndef mean_rgb_from_roi(frame, roi):\n    x, y, w, h = roi\n    roi_frame = frame[y:y+h, x:x+w, :]\n\n    if roi_frame.size == 0:\n        return None\n\n    mean_bgr = np.mean(roi_frame.reshape(-1, 3), axis=0)\n    mean_rgb = mean_bgr[::-1]\n\n    return mean_rgb\n\n\n# =========================\n# POS algorithm\n# =========================\ndef pos_rppg(rgb_signal, fs, window_seconds=1.6):\n    rgb_signal = np.asarray(rgb_signal, dtype=float)\n    n = rgb_signal.shape[0]\n\n    if n < int(window_seconds * fs):\n        return np.zeros(n)\n\n    window_length = int(window_seconds * fs)\n    h = np.zeros(n)\n\n    for start in range(0, n - window_length + 1):\n        end = start + window_length\n        C = rgb_signal[start:end, :].T\n\n        mean_color = np.mean(C, axis=1, keepdims=True)\n        mean_color[mean_color == 0] = 1e-8\n\n        Cn = C / mean_color\n\n        S1 = Cn[1, :] - Cn[2, :]\n        S2 = Cn[1, :] + Cn[2, :] - 2 * Cn[0, :]\n\n        std_s2 = np.std(S2)\n\n        if std_s2 < 1e-8:\n            alpha = 0.0\n        else:\n            alpha = np.std(S1) / std_s2\n\n        H = S1 + alpha * S2\n        H = H - np.mean(H)\n\n        h[start:end] += H\n\n    return h\n\n\n# =========================\n# Load video\n# =========================\ncap = cv2.VideoCapture(video_path)\n\nif not cap.isOpened():\n    raise RuntimeError("Video could not be opened.")\n\nfps = cap.get(cv2.CAP_PROP_FPS)\n\nif fps is None or fps <= 0:\n    fps = 20.0\n\nprint("Video FPS:", fps)\n\nface_cascade = cv2.CascadeClassifier(\n    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"\n)\n\n\n# =========================\n# Extract multi-ROI RGB signal\n# =========================\ntimes = []\nrgb_signal = []\ngreen_signal = []\npreview_frames = []\n\nframe_index = 0\n\nwhile True:\n    ret, frame = cap.read()\n\n    if not ret:\n        break\n\n    current_time = frame_index / fps\n    frame_index += 1\n\n    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)\n\n    faces = face_cascade.detectMultiScale(\n        gray,\n        scaleFactor=1.1,\n        minNeighbors=3,\n        minSize=(100, 100)\n    )\n\n    if len(faces) == 0:\n        continue\n\n    faces = sorted(faces, key=lambda f: f[2] * f[3], reverse=True)\n    face = faces[0]\n    x, y, w, h = face\n\n    rois = get_multi_rois(face, frame.shape)\n\n    rgb_values = []\n\n    for roi_name, roi in rois.items():\n        mean_rgb = mean_rgb_from_roi(frame, roi)\n        if mean_rgb is not None:\n            rgb_values.append(mean_rgb)\n\n    if len(rgb_values) == 0:\n        continue\n\n    mean_rgb_all = np.mean(np.asarray(rgb_values), axis=0)\n\n    times.append(current_time)\n    rgb_signal.append(mean_rgb_all)\n    green_signal.append(mean_rgb_all[1])\n\n    if len(preview_frames) < 5:\n        frame_preview = frame.copy()\n\n        cv2.rectangle(\n            frame_preview,\n            (x, y),\n            (x + w, y + h),\n            (0, 255, 0),\n            2\n        )\n\n        colors = {\n            "forehead": (0, 0, 255),\n            "left_cheek": (255, 0, 0),\n            "right_cheek": (255, 0, 0)\n        }\n\n        for roi_name, roi in rois.items():\n            rx, ry, rw, rh = roi\n            cv2.rectangle(\n                frame_preview,\n                (rx, ry),\n                (rx + rw, ry + rh),\n                colors.get(roi_name, (0, 0, 255)),\n                2\n            )\n            cv2.putText(\n                frame_preview,\n                roi_name,\n                (rx, max(0, ry - 5)),\n                cv2.FONT_HERSHEY_SIMPLEX,\n                0.5,\n                colors.get(roi_name, (0, 0, 255)),\n                1\n            )\n\n        preview_frames.append(frame_preview)\n\ncap.release()\n\ntimes = np.asarray(times)\nrgb_signal = np.asarray(rgb_signal)\ngreen_signal = np.asarray(green_signal)\n\nprint("Extracted samples:", len(rgb_signal))\n\nif len(rgb_signal) < 60:\n    raise RuntimeError("Not enough valid samples. Try clearer video, better lighting, or larger face.")\n\n\n# =========================\n# Preview frames\n# =========================\nprint("Preview frames:")\nprint("Green rectangle = face")\nprint("Red rectangle = forehead")\nprint("Blue rectangles = cheeks")\n\nfor frame in preview_frames:\n    cv2_imshow(frame)\n\n\n# =========================\n# Sampling rate\n# =========================\nduration = times[-1] - times[0]\n\nif duration <= 0:\n    raise RuntimeError("Invalid timestamps.")\n\nsampling_rate = (len(times) - 1) / duration\n\nprint("Estimated sampling rate:", sampling_rate)\n\n\n# =========================\n# Process signals\n# =========================\ngreen_norm = normalize_signal(green_signal)\n\ngreen_filt = bandpass_filter(\n    green_norm,\n    fs=sampling_rate,\n    min_bpm=MIN_BPM,\n    max_bpm=MAX_BPM\n)\n\npos_signal = pos_rppg(\n    rgb_signal,\n    fs=sampling_rate,\n    window_seconds=POS_WINDOW_SECONDS\n)\n\npos_signal = detrend(pos_signal)\npos_norm = normalize_signal(pos_signal)\n\npos_filt = bandpass_filter(\n    pos_norm,\n    fs=sampling_rate,\n    min_bpm=MIN_BPM,\n    max_bpm=MAX_BPM\n)\n\n\n# =========================\n# Smart FFT estimates\n# =========================\nhr_green_fft, hr_pos_fft, hr_consensus, green_info, pos_info = estimate_smart_fft_pair(\n    green_filt,\n    pos_filt,\n    sampling_rate,\n    min_bpm=MIN_BPM,\n    max_bpm=MAX_BPM,\n    trim_start_seconds=TRIM_START_SECONDS\n)\n\n\n# =========================\n# Peak estimates\n# =========================\nhr_green_peak = estimate_hr_peaks(\n    green_filt,\n    sampling_rate,\n    min_bpm=MIN_BPM,\n    max_bpm=MAX_BPM\n)\n\nhr_pos_peak = estimate_hr_peaks(\n    pos_filt,\n    sampling_rate,\n    min_bpm=MIN_BPM,\n    max_bpm=MAX_BPM\n)\n\n\n# =========================\n# Adaptive final correction\n# =========================\nhr_green_fft, hr_pos_fft, hr_consensus, peak_guidance_hr = adaptive_final_correction(\n    green_info,\n    pos_info,\n    hr_green_fft,\n    hr_pos_fft,\n    hr_consensus,\n    hr_green_peak,\n    hr_pos_peak\n)\n\n\n# =========================\n# Motion-mode correction\n# =========================\nhr_green_fft, hr_pos_fft, hr_consensus, motion_peak_corrected = motion_peak_guidance_correction(\n    hr_green_fft,\n    hr_pos_fft,\n    hr_consensus,\n    peak_guidance_hr\n)\n\n\n# =========================\n# Results\n# =========================\nprint("\\n===== Smart Green-only rPPG Results =====")\nprint(f"HR Green FFT: {hr_green_fft:.2f} bpm" if hr_green_fft is not None else "HR Green FFT: Not available")\nprint(f"HR Green Peak: {hr_green_peak:.2f} bpm" if hr_green_peak is not None else "HR Green Peak: Not available")\n\nprint("\\n===== Smart POS rPPG Results =====")\nprint(f"HR POS FFT: {hr_pos_fft:.2f} bpm" if hr_pos_fft is not None else "HR POS FFT: Not available")\nprint(f"HR POS Peak: {hr_pos_peak:.2f} bpm" if hr_pos_peak is not None else "HR POS Peak: Not available")\n\nprint("\\n===== Consensus Result =====")\nprint(f"Final Consensus HR: {hr_consensus:.2f} bpm" if hr_consensus is not None else "Final Consensus HR: Not available")\nprint(f"Peak Guidance HR: {peak_guidance_hr:.2f} bpm" if peak_guidance_hr is not None else "Peak Guidance HR: Not available")\nprint(f"Motion Peak Correction Applied: {motion_peak_corrected}")\n\n\n# =========================\n# Debug information\n# =========================\ndef print_candidates(title, info):\n    print("\\n" + title)\n\n    print("Full best:", info["full_best"])\n    print("Window HR:", info["window_hr"])\n    print("Window median:", info["window_median"])\n    print("Window MAD:", info["window_mad"])\n    print("Reliability:", info["reliability"])\n\n    print("Top FFT candidates:")\n    for i, c in enumerate(info["candidates"]):\n        print(\n            f"{i + 1}. {c[\'bpm\']:.2f} bpm | "\n            f"mag={c[\'magnitude\']:.3f} | "\n            f"norm={c[\'norm_magnitude\']:.3f}"\n        )\n\n\nprint_candidates("Green FFT Debug", green_info)\nprint_candidates("POS FFT Debug", pos_info)\n\n\n# =========================\n# Plot signals\n# =========================\nrelative_time = times - times[0]\n\nplt.figure(figsize=(14, 5))\nplt.plot(relative_time, green_norm, label="Green-only normalized", alpha=0.35)\nplt.plot(relative_time, green_filt, label="Green-only filtered", linewidth=2)\nplt.axvline(TRIM_START_SECONDS, linestyle="--", label="FFT trim start")\nplt.xlabel("Time (s)")\nplt.ylabel("Amplitude")\nplt.title("Green-only Baseline rPPG")\nplt.grid(True)\nplt.legend()\nplt.show()\n\nplt.figure(figsize=(14, 5))\nplt.plot(relative_time, pos_norm, label="POS normalized", alpha=0.35)\nplt.plot(relative_time, pos_filt, label="POS filtered", linewidth=2)\nplt.axvline(TRIM_START_SECONDS, linestyle="--", label="FFT trim start")\nplt.xlabel("Time (s)")\nplt.ylabel("Amplitude")\nplt.title("POS rPPG Signal")\nplt.grid(True)\nplt.legend()\nplt.show()\n\n\n# =========================\n# Plot FFT comparison\n# =========================\ngreen_bpm_axis, green_spectrum = compute_fft_spectrum(\n    green_filt,\n    sampling_rate,\n    min_bpm=MIN_BPM,\n    max_bpm=MAX_BPM,\n    trim_start_seconds=TRIM_START_SECONDS,\n    zero_padding_factor=FFT_ZERO_PADDING_FACTOR\n)\n\npos_bpm_axis, pos_spectrum = compute_fft_spectrum(\n    pos_filt,\n    sampling_rate,\n    min_bpm=MIN_BPM,\n    max_bpm=MAX_BPM,\n    trim_start_seconds=TRIM_START_SECONDS,\n    zero_padding_factor=FFT_ZERO_PADDING_FACTOR\n)\n\nplt.figure(figsize=(14, 5))\n\nif green_bpm_axis is not None and green_spectrum is not None:\n    plt.plot(\n        green_bpm_axis,\n        green_spectrum,\n        label="Green-only spectrum",\n        alpha=0.7\n    )\n\nif pos_bpm_axis is not None and pos_spectrum is not None:\n    plt.plot(\n        pos_bpm_axis,\n        pos_spectrum,\n        label="POS spectrum",\n        linewidth=2\n    )\n\nif hr_green_fft is not None:\n    plt.axvline(hr_green_fft, linestyle="--", label=f"Green FFT = {hr_green_fft:.2f}")\n\nif hr_pos_fft is not None:\n    plt.axvline(hr_pos_fft, linestyle="--", label=f"POS FFT = {hr_pos_fft:.2f}")\n\nif hr_consensus is not None:\n    plt.axvline(hr_consensus, linestyle=":", label=f"Consensus = {hr_consensus:.2f}")\n\nif peak_guidance_hr is not None:\n    plt.axvline(peak_guidance_hr, linestyle="-.", label=f"Peak guidance = {peak_guidance_hr:.2f}")\n\nplt.xlabel("Heart Rate (BPM)")\nplt.ylabel("FFT Magnitude")\nplt.title("Smart FFT Spectrum Comparison: Green-only vs POS")\nplt.grid(True)\nplt.legend()\nplt.show()\n\n\n# =========================\n# Peak detection visualization for POS\n# =========================\npeak_distance = max(1, int(sampling_rate * 60.0 / MAX_BPM))\npeak_prominence = max(0.35 * np.std(pos_filt), 0.15)\n\npos_peaks, _ = find_peaks(\n    pos_filt - np.mean(pos_filt),\n    distance=peak_distance,\n    prominence=peak_prominence\n)\n\nplt.figure(figsize=(14, 5))\nplt.plot(relative_time, pos_filt, label="POS filtered")\nplt.plot(relative_time[pos_peaks], pos_filt[pos_peaks], "x", label="Detected POS peaks")\nplt.axvline(TRIM_START_SECONDS, linestyle="--", label="FFT trim start")\nplt.xlabel("Time (s)")\nplt.ylabel("Amplitude")\nplt.title("Peak Detection on POS rPPG Signal")\nplt.grid(True)\nplt.legend()\nplt.show()'


# =========================
# Master execution
# =========================
print("Upload one video. The code will first classify the head motion type, then run the matching rPPG pipeline.")
uploaded = files.upload()
video_path = list(uploaded.keys())[0]
print("Uploaded video:", video_path)

print("\nRunning head motion classifier...")
classification_data = extract_head_motion(video_path)
classification_features = compute_features(classification_data)
predicted_class, confidence = classify_motion(classification_features)

if SHOW_PREVIEW_FRAMES:
    print("\nHead tracking preview frames:")
    print("Green box = detected face")
    print("Red dot = tracked head center")
    for frame in classification_data["preview_frames"]:
        cv2_imshow(frame)

print("\n" + "=" * 60)
print("HEAD MOTION CLASSIFICATION RESULT")
print("=" * 60)
print("Video:", video_path)
print("Predicted class:", predicted_class)
print(f"Confidence: {confidence:.2f}")
print("\nMotion features:")
for key, value in classification_features.items():
    if isinstance(value, float):
        print(f"{key}: {value:.4f}")
    else:
        print(f"{key}: {value}")

print("\n" + "=" * 60)
print("RUNNING SELECTED rPPG PIPELINE")
print("=" * 60)

if predicted_class == "stable":
    print("Selected pipeline: FIXED / STABLE")
    exec(FIXED_PIPELINE_CODE, globals())
elif predicted_class == "left_right":
    print("Selected pipeline: LEFT-RIGHT")
    exec(LEFT_RIGHT_PIPELINE_CODE, globals())
elif predicted_class == "zigzag":
    print("Selected pipeline: ZIGZAG")
    exec(ZIGZAG_PIPELINE_CODE, globals())
else:
    print("Unknown class, using fixed pipeline as fallback.")
    exec(FIXED_PIPELINE_CODE, globals())
